# Teaching an Object Detector Not to Trust the Background

### FathomNet-CLEF 2026 — a positive-unlabeled marine object detection pipeline

This notebook is the executable form of the five-stage pipeline:

| Stage | Idea | Where |
|---|---|---|
| 1 | Clean the supervision before changing the network | §3–§4 |
| 2 | One high-resolution RT-DETR detector | §6–§7 |
| 3 | Conservative pseudo-label recovery | §8 |
| 4 | PU-aware background loss | §5 |
| 5 | Two-scale inference + Soft-NMS | §9 |

**The learning problem.** Training annotations are deliberately incomplete: an
image may contain a crab, two fish and an urchin while only the crab is boxed.
Ordinary detection training reads every unannotated region as background, so the
model is punished for finding organisms that are genuinely there. Evaluation, by
contrast, is fully annotated. The pipeline below is built around one question:

> How do we train a detector when "not labeled" does not mean "negative"?

---

### How to run this on Kaggle

Set `CFG.stage` in the config cell and run top to bottom. Stages are separate so
the work fits inside Kaggle's session limit and GPU quota:

| `CFG.stage` | What it does | Needs GPU | Rough time |
|---|---|---|---|
| `"verify"` | Runs the built-in test suite only | no | seconds |
| `"audit"` | Stage 1: audit + clean + split, no training | no | ~1 min |
| `"train_base"` | Stage 2: baseline detector → checkpoint | **yes** | hours |
| `"harvest"` | Stage 3: pseudo-labels from the baseline | **yes** | ~30 min |
| `"train_pu"` | Stages 3+4: retrain with pseudo-labels + PU loss | **yes** | hours |
| `"infer"` | Stage 5: two-scale inference + submission | **yes** | ~30 min |

**Before a GPU stage:** turn on *Settings → Accelerator → GPU*, and
*Settings → Internet → On* (needed to fetch RT-DETR pretrained weights and, unless
you have attached an images dataset, the imagery itself).

**Start with `"verify"`, then `"audit"`.** Both run on CPU in under a minute and
will catch a wrong dataset path or a broken assumption before you spend GPU quota.

> **Scope note.** The numbers this notebook produces depend entirely on your run.
> Nothing here is a reported leaderboard result, and the defaults below are
> starting points chosen to be safe, not tuned optima.

## 1. Configuration

Every knob in one place. The PU-specific ones are grouped at the bottom and are
the only ones worth sweeping first.

In [ ]:
import os, sys, json, math, time, random, collections, copy, hashlib, warnings
from dataclasses import dataclass, field, asdict
from pathlib import Path

import numpy as np

warnings.filterwarnings("ignore", category=UserWarning)


@dataclass
class CFG:
    # ---- which stage to run -------------------------------------------------
    # "verify" | "audit" | "train_base" | "harvest" | "train_pu" | "infer"
    stage: str = "verify"

    # ---- paths --------------------------------------------------------------
    # Kaggle mounts competition data under /kaggle/input at an unpredictable
    # depth, so the annotation files are discovered rather than hardcoded.
    input_roots: tuple = ("/kaggle/input", "./data", ".")
    work_dir: str = "/kaggle/working" if Path("/kaggle/working").exists() else "./work"
    image_dir: str = ""        # set automatically; where frames are cached

    # ---- data ---------------------------------------------------------------
    num_classes: int = 32
    val_fraction: float = 0.15
    split_salt: str = "fathomnet-pu-v1"
    split_group_key: str = ""  # e.g. "source" if images carry a provenance field
    drop_tiny_boxes: bool = False

    # Cap the number of images actually used. Keep this small for a first
    # end-to-end pass; set to 0 for the full set.
    max_images: int = 200
    download_workers: int = 16

    # ---- model / training ---------------------------------------------------
    model_name: str = "PekingU/rtdetr_r50vd_coco_o365"
    img_size: int = 800          # Stage 2: the high-resolution training path
    batch_size: int = 2
    grad_accum: int = 4
    epochs: int = 2
    lr: float = 1e-4
    lr_backbone: float = 1e-5
    weight_decay: float = 1e-4
    clip_grad: float = 0.1
    num_workers: int = 2
    amp: bool = True
    seed: int = 1337

    # ---- Stage 3: pseudo-label recovery ------------------------------------
    pseudo_score_threshold: float = 0.60
    pseudo_gt_iou_threshold: float = 0.50
    pseudo_consistency_iou: float = 0.60
    pseudo_min_views: int = 2
    pseudo_max_per_image: int = 5

    # ---- Stage 4: PU-aware background loss ---------------------------------
    # The single most important setting in the notebook. pu_w_min must stay > 0:
    # at 0 the model gets no background supervision on its confident mistakes and
    # every rock and sediment texture becomes an organism.
    pu_w_min: float = 0.25
    pu_w_max: float = 1.00
    pu_tau: float = 0.50
    pu_temperature: float = 0.10
    pu_disabled: bool = False     # True = ordinary background loss (ablation)
    pseudo_label_weight: float = 0.50

    # ---- Stage 5: two-scale inference --------------------------------------
    infer_scales: tuple = (640, 960)
    infer_scale_weights: tuple = (1.0, 1.0)
    softnms_method: str = "gaussian"
    softnms_sigma: float = 0.50
    softnms_iou_threshold: float = 0.30
    score_threshold: float = 0.01
    max_dets_per_image: int = 100


CFG = CFG()

# Allow the stage to be overridden by an env var, handy when scheduling runs.
CFG.stage = os.environ.get("PU_STAGE", CFG.stage)

Path(CFG.work_dir).mkdir(parents=True, exist_ok=True)

random.seed(CFG.seed)
np.random.seed(CFG.seed)

print(f"stage     : {CFG.stage}")
print(f"work_dir  : {CFG.work_dir}")

## 2. Environment probe

Establish what is actually available before relying on it. A stage that needs a
GPU stops here with a clear message rather than failing thirty minutes in.

In [ ]:
def probe_environment():
    info = {}
    try:
        import torch
        info["torch"] = torch.__version__
        info["cuda"] = torch.cuda.is_available()
        info["n_gpu"] = torch.cuda.device_count() if torch.cuda.is_available() else 0
        if info["cuda"]:
            info["gpu_name"] = torch.cuda.get_device_name(0)
            info["gpu_mem_gb"] = round(
                torch.cuda.get_device_properties(0).total_memory / 1024**3, 1
            )
    except Exception as e:
        info["torch"] = f"MISSING ({e})"
        info["cuda"] = False
        info["n_gpu"] = 0

    for mod in ("torchvision", "transformers", "pycocotools", "scipy", "PIL"):
        try:
            m = __import__(mod)
            info[mod] = getattr(m, "__version__", "present")
        except Exception:
            info[mod] = "MISSING"

    info["on_kaggle"] = Path("/kaggle").exists()
    info["internet"] = False
    try:
        import urllib.request
        urllib.request.urlopen("https://huggingface.co", timeout=8)
        info["internet"] = True
    except Exception:
        pass
    return info


ENV = probe_environment()
for k, v in ENV.items():
    print(f"{k:14s}: {v}")

GPU_STAGES = {"train_base", "harvest", "train_pu", "infer"}
if CFG.stage in GPU_STAGES and not ENV.get("cuda"):
    print(
        "\n>>> STOP: stage "
        f"'{CFG.stage}' needs a GPU but none is visible.\n"
        ">>> Settings -> Accelerator -> GPU, then re-run.\n"
        ">>> ('verify' and 'audit' run fine on CPU.)"
    )
if CFG.stage in GPU_STAGES and not ENV.get("internet"):
    print(
        "\n>>> WARNING: no internet. Pretrained RT-DETR weights cannot be fetched\n"
        ">>> and imagery cannot be downloaded. Settings -> Internet -> On."
    )

## 3. Core ops

The post-processing and data-hygiene algorithms. These are deliberately NumPy-only:
they carry the pipeline's actual reasoning, they run identically on CPU, and they
are covered by the test suite in §10 — so a mistake here shows up in seconds
rather than after an epoch of training.

Box convention: `xywh` is COCO-native `[x, y, w, h]`; `xyxy` is `[x1, y1, x2, y2]`.

In [ ]:
"""Post-processing and data-hygiene ops for positive-unlabeled detection.

NumPy-only so they stay testable on CPU. Boxes are xyxy float arrays of shape
(N, 4) unless a function says otherwise; "xywh" means COCO-native
[x_min, y_min, width, height].
"""


import numpy as np

# --------------------------------------------------------------------------- #
# Box conversions
# --------------------------------------------------------------------------- #


def xywh_to_xyxy(boxes: np.ndarray) -> np.ndarray:
    """COCO [x, y, w, h] -> [x1, y1, x2, y2]."""
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
    out = np.empty_like(boxes)
    out[:, 0] = boxes[:, 0]
    out[:, 1] = boxes[:, 1]
    out[:, 2] = boxes[:, 0] + boxes[:, 2]
    out[:, 3] = boxes[:, 1] + boxes[:, 3]
    return out


def xyxy_to_xywh(boxes: np.ndarray) -> np.ndarray:
    """[x1, y1, x2, y2] -> COCO [x, y, w, h]."""
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
    out = np.empty_like(boxes)
    out[:, 0] = boxes[:, 0]
    out[:, 1] = boxes[:, 1]
    out[:, 2] = boxes[:, 2] - boxes[:, 0]
    out[:, 3] = boxes[:, 3] - boxes[:, 1]
    return out


def box_areas(boxes: np.ndarray) -> np.ndarray:
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
    w = np.clip(boxes[:, 2] - boxes[:, 0], 0.0, None)
    h = np.clip(boxes[:, 3] - boxes[:, 1], 0.0, None)
    return w * h


def iou_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """
    Pairwise IoU between two sets of xyxy boxes.

    Returns (len(a), len(b)). Zero-area boxes yield IoU 0 rather than NaN.
    """
    a = np.asarray(a, dtype=np.float64).reshape(-1, 4)
    b = np.asarray(b, dtype=np.float64).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float64)

    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])

    inter = np.clip(x2 - x1, 0.0, None) * np.clip(y2 - y1, 0.0, None)
    union = box_areas(a)[:, None] + box_areas(b)[None, :] - inter
    with np.errstate(divide="ignore", invalid="ignore"):
        iou = np.where(union > 0.0, inter / union, 0.0)
    return iou


# --------------------------------------------------------------------------- #
# Stage 5a: Soft-NMS
# --------------------------------------------------------------------------- #


def soft_nms(
    boxes: np.ndarray,
    scores: np.ndarray,
    method: str = "gaussian",
    sigma: float = 0.5,
    iou_threshold: float = 0.3,
    score_threshold: float = 1e-3,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Soft-NMS (Bodla et al., 2017), single class.

    Rather than deleting an overlapping box outright, its score is *decayed* by
    the amount of overlap. This matters in dense marine scenes where two real
    organisms genuinely overlap: hard NMS would erase the weaker one.

      gaussian: s_i <- s_i * exp(-iou^2 / sigma)          (applied to all overlaps)
      linear:   s_i <- s_i * (1 - iou)   for iou > iou_threshold

    Returns (kept_boxes, kept_scores, kept_indices_into_input).
    Boxes whose decayed score falls below `score_threshold` are dropped.
    """
    if method not in {"gaussian", "linear", "hard"}:
        raise ValueError(f"unknown method {method!r}")

    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4).copy()
    scores = np.asarray(scores, dtype=np.float64).reshape(-1).copy()
    if len(boxes) != len(scores):
        raise ValueError("boxes and scores length mismatch")
    if len(boxes) == 0:
        return boxes, scores, np.zeros(0, dtype=np.int64)

    idx = np.arange(len(boxes))
    keep_b, keep_s, keep_i = [], [], []

    while len(scores) > 0:
        # Take the current maximum as the "anchor" detection.
        m = int(np.argmax(scores))
        b_m, s_m, i_m = boxes[m].copy(), float(scores[m]), int(idx[m])

        if s_m < score_threshold:
            break

        keep_b.append(b_m)
        keep_s.append(s_m)
        keep_i.append(i_m)

        # Remove the anchor, then decay whatever overlaps it.
        boxes = np.delete(boxes, m, axis=0)
        scores = np.delete(scores, m, axis=0)
        idx = np.delete(idx, m, axis=0)
        if len(boxes) == 0:
            break

        ious = iou_matrix(b_m[None, :], boxes)[0]

        if method == "gaussian":
            scores = scores * np.exp(-(ious**2) / sigma)
        elif method == "linear":
            decay = np.where(ious > iou_threshold, 1.0 - ious, 1.0)
            scores = scores * decay
        else:  # hard NMS, for ablation
            scores = np.where(ious > iou_threshold, 0.0, scores)

        surviving = scores >= score_threshold
        boxes, scores, idx = boxes[surviving], scores[surviving], idx[surviving]

    if not keep_b:
        return (
            np.zeros((0, 4), dtype=np.float64),
            np.zeros(0, dtype=np.float64),
            np.zeros(0, dtype=np.int64),
        )
    return (
        np.stack(keep_b),
        np.asarray(keep_s, dtype=np.float64),
        np.asarray(keep_i, dtype=np.int64),
    )


def soft_nms_per_class(
    boxes: np.ndarray,
    scores: np.ndarray,
    labels: np.ndarray,
    **kw,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Apply Soft-NMS independently within each class.

    Cross-class suppression is wrong here: a shrimp sitting on a sponge produces
    two heavily-overlapping boxes of different classes and both are correct.
    """
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
    scores = np.asarray(scores, dtype=np.float64).reshape(-1)
    labels = np.asarray(labels).reshape(-1)

    out_b, out_s, out_l = [], [], []
    for c in np.unique(labels):
        sel = np.flatnonzero(labels == c)
        kb, ks, _ = soft_nms(boxes[sel], scores[sel], **kw)
        if len(kb):
            out_b.append(kb)
            out_s.append(ks)
            out_l.append(np.full(len(kb), c, dtype=labels.dtype))

    if not out_b:
        return (
            np.zeros((0, 4), dtype=np.float64),
            np.zeros(0, dtype=np.float64),
            np.zeros(0, dtype=labels.dtype),
        )

    b = np.concatenate(out_b)
    s = np.concatenate(out_s)
    l = np.concatenate(out_l)
    order = np.argsort(-s)  # return in descending-score order
    return b[order], s[order], l[order]


# --------------------------------------------------------------------------- #
# Stage 5b: two-scale inference merge
# --------------------------------------------------------------------------- #


def scale_boxes(boxes: np.ndarray, from_size, to_size) -> np.ndarray:
    """
    Map boxes between two image sizes. Sizes are (width, height).

    Used to bring detections produced at an inference resolution back into the
    coordinate frame of the original frame, which is what the metric is computed in.
    """
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
    fw, fh = float(from_size[0]), float(from_size[1])
    tw, th = float(to_size[0]), float(to_size[1])
    if fw <= 0 or fh <= 0:
        raise ValueError("from_size must be positive")
    sx, sy = tw / fw, th / fh
    out = boxes.copy()
    out[:, [0, 2]] *= sx
    out[:, [1, 3]] *= sy
    return out


def clip_boxes(boxes: np.ndarray, size) -> np.ndarray:
    """Clip xyxy boxes to an image of size (width, height)."""
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4).copy()
    w, h = float(size[0]), float(size[1])
    boxes[:, 0] = np.clip(boxes[:, 0], 0.0, w)
    boxes[:, 1] = np.clip(boxes[:, 1], 0.0, h)
    boxes[:, 2] = np.clip(boxes[:, 2], 0.0, w)
    boxes[:, 3] = np.clip(boxes[:, 3], 0.0, h)
    return boxes


def merge_multiscale(
    passes: list[dict],
    original_size,
    method: str = "gaussian",
    sigma: float = 0.5,
    iou_threshold: float = 0.3,
    score_threshold: float = 1e-3,
) -> dict:
    """
    Merge detections from several inference passes into one set.

    Each entry of `passes` is a dict:
        {"boxes": (N,4) xyxy, "scores": (N,), "labels": (N,),
         "size": (w, h) the resolution those boxes are expressed in,
         "weight": optional float score multiplier for this pass}

    Every pass is mapped back into `original_size`, concatenated, clipped, and
    reconciled with per-class Soft-NMS.

    The medium-resolution pass contributes scene context; the high-resolution
    pass recovers small organisms. `weight` lets you trust one pass more than
    the other without retraining anything.
    """
    all_b, all_s, all_l = [], [], []
    for p in passes:
        b = np.asarray(p["boxes"], dtype=np.float64).reshape(-1, 4)
        s = np.asarray(p["scores"], dtype=np.float64).reshape(-1)
        l = np.asarray(p["labels"]).reshape(-1)
        if len(b) == 0:
            continue
        b = scale_boxes(b, p["size"], original_size)
        b = clip_boxes(b, original_size)
        s = s * float(p.get("weight", 1.0))
        all_b.append(b)
        all_s.append(s)
        all_l.append(l)

    if not all_b:
        return {
            "boxes": np.zeros((0, 4)),
            "scores": np.zeros(0),
            "labels": np.zeros(0, dtype=np.int64),
        }

    b = np.concatenate(all_b)
    s = np.concatenate(all_s)
    l = np.concatenate(all_l)

    # Drop degenerate boxes produced by clipping at the frame edge.
    valid = box_areas(b) > 0
    b, s, l = b[valid], s[valid], l[valid]

    b, s, l = soft_nms_per_class(
        b,
        s,
        l,
        method=method,
        sigma=sigma,
        iou_threshold=iou_threshold,
        score_threshold=score_threshold,
    )
    return {"boxes": b, "scores": s, "labels": l}


# --------------------------------------------------------------------------- #
# Stage 4: PU-aware background weight (NumPy reference of the torch version)
# --------------------------------------------------------------------------- #


def pu_background_weight(
    objectness: np.ndarray,
    w_min: float = 0.25,
    w_max: float = 1.0,
    tau: float = 0.5,
    temperature: float = 0.1,
) -> np.ndarray:
    """
    Weight w_bg(q) applied to the background loss of an *unmatched* prediction.

    The blog's Stage 4 in one function. An unmatched query with weak object
    evidence is almost certainly background, so it keeps the full penalty. An
    unmatched query with *strong* object evidence may be a missing annotation,
    so its background penalty is softened -- the model is allowed to say
    "I am less certain that this region is negative."

    Smooth sigmoid interpolation between w_max (probably background) and
    w_min (possibly a missing positive):

        w(q) = w_max - (w_max - w_min) * sigmoid((objectness - tau) / temperature)

    Critically w_min > 0. Setting w_min = 0 removes background supervision for
    confident predictions entirely and false positives explode -- every rock and
    sediment texture becomes an organism. This is the single most important
    hyperparameter in the pipeline.
    """
    if not (0.0 <= w_min <= w_max):
        raise ValueError("require 0 <= w_min <= w_max")
    if temperature <= 0:
        raise ValueError("temperature must be > 0")

    q = np.asarray(objectness, dtype=np.float64)
    gate = 1.0 / (1.0 + np.exp(-(q - tau) / temperature))
    return w_max - (w_max - w_min) * gate


# --------------------------------------------------------------------------- #
# Stage 3: conservative pseudo-label recovery
# --------------------------------------------------------------------------- #


def dedup_against_gt(
    boxes: np.ndarray,
    labels: np.ndarray,
    gt_boxes: np.ndarray,
    gt_labels: np.ndarray,
    iou_threshold: float = 0.5,
    class_aware: bool = True,
) -> np.ndarray:
    """
    Boolean mask of candidates that do NOT already duplicate a ground-truth box.

    With class_aware=True a candidate is only suppressed by a GT box of the same
    class, so a shrimp detected on top of an annotated sponge survives.
    """
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
    labels = np.asarray(labels).reshape(-1)
    gt_boxes = np.asarray(gt_boxes, dtype=np.float64).reshape(-1, 4)
    gt_labels = np.asarray(gt_labels).reshape(-1)

    if len(boxes) == 0:
        return np.zeros(0, dtype=bool)
    if len(gt_boxes) == 0:
        return np.ones(len(boxes), dtype=bool)

    ious = iou_matrix(boxes, gt_boxes)
    if class_aware:
        same = labels[:, None] == gt_labels[None, :]
        ious = np.where(same, ious, 0.0)
    return ious.max(axis=1) < iou_threshold


def consistency_filter(
    view_dets: list[dict],
    iou_threshold: float = 0.6,
    min_views: int = 2,
    reference: int = 0,
) -> np.ndarray:
    """
    Keep only reference-view candidates corroborated across TTA views.

    `view_dets` is a list of dicts {"boxes", "labels"}, one per view, ALL already
    mapped back into the original coordinate frame. A reference candidate counts
    a view as agreeing if that view holds a same-class box at IoU >= threshold.

    "One confident prediction = interesting; one confident AND stable prediction
    = much more useful." A texture artifact rarely survives a flip and a resize
    at the same location with the same class; a real organism usually does.

    Returns a boolean mask over the reference view's boxes.
    """
    if not view_dets:
        return np.zeros(0, dtype=bool)
    if not (0 <= reference < len(view_dets)):
        raise IndexError("reference view out of range")

    ref = view_dets[reference]
    ref_b = np.asarray(ref["boxes"], dtype=np.float64).reshape(-1, 4)
    ref_l = np.asarray(ref["labels"]).reshape(-1)
    if len(ref_b) == 0:
        return np.zeros(0, dtype=bool)

    # The reference view always corroborates itself.
    votes = np.ones(len(ref_b), dtype=np.int64)

    for vi, v in enumerate(view_dets):
        if vi == reference:
            continue
        vb = np.asarray(v["boxes"], dtype=np.float64).reshape(-1, 4)
        vl = np.asarray(v["labels"]).reshape(-1)
        if len(vb) == 0:
            continue
        ious = iou_matrix(ref_b, vb)
        same = ref_l[:, None] == vl[None, :]
        agree = (np.where(same, ious, 0.0) >= iou_threshold).any(axis=1)
        votes += agree.astype(np.int64)

    return votes >= min_views


def harvest_pseudo_labels(
    view_dets: list[dict],
    gt_boxes: np.ndarray,
    gt_labels: np.ndarray,
    score_threshold: float = 0.6,
    gt_iou_threshold: float = 0.5,
    consistency_iou: float = 0.6,
    min_views: int = 2,
    max_per_image: int | None = 5,
) -> dict:
    """
    The full Stage 3 funnel for one image, in the blog's order:

        confidence filter -> IoU dedup against GT -> cross-view consistency -> cap

    A pseudo-label is only worth adding if it is more likely to *correct* missing
    supervision than to *inject* a new error, so every stage here removes
    candidates and none adds any. `max_per_image` is a blunt but effective guard
    against one pathological frame dumping dozens of pseudo-labels into training.

    Returns the surviving boxes/scores/labels plus a per-stage survivor count,
    which is what you actually watch to tune the thresholds.
    """
    ref = view_dets[0]
    boxes = np.asarray(ref["boxes"], dtype=np.float64).reshape(-1, 4)
    scores = np.asarray(ref["scores"], dtype=np.float64).reshape(-1)
    labels = np.asarray(ref["labels"]).reshape(-1)

    counts = {"candidates": int(len(boxes))}

    # 1. confidence
    keep = scores >= score_threshold
    boxes, scores, labels = boxes[keep], scores[keep], labels[keep]
    counts["after_confidence"] = int(len(boxes))

    # 2. dedup against existing ground truth
    if len(boxes):
        keep = dedup_against_gt(
            boxes, labels, gt_boxes, gt_labels, iou_threshold=gt_iou_threshold
        )
        boxes, scores, labels = boxes[keep], scores[keep], labels[keep]
    counts["after_gt_dedup"] = int(len(boxes))

    # 3. cross-view consistency, evaluated on the surviving subset
    if len(boxes) and len(view_dets) > 1:
        views = [{"boxes": boxes, "labels": labels}] + [
            {"boxes": v["boxes"], "labels": v["labels"]} for v in view_dets[1:]
        ]
        keep = consistency_filter(
            views,
            iou_threshold=consistency_iou,
            min_views=min_views,
            reference=0,
        )
        boxes, scores, labels = boxes[keep], scores[keep], labels[keep]
    counts["after_consistency"] = int(len(boxes))

    # 4. cap, keeping the most confident
    if max_per_image is not None and len(boxes) > max_per_image:
        order = np.argsort(-scores)[:max_per_image]
        boxes, scores, labels = boxes[order], scores[order], labels[order]
    counts["kept"] = int(len(boxes))

    return {"boxes": boxes, "scores": scores, "labels": labels, "counts": counts}

## 4. Stage 1 — clean the supervision before changing the network

In a positive-unlabeled setting an annotation *bug* and a deliberately *missing*
label look identical downstream, and only one of the two is yours to fix. So the
audit runs first and is read-only; cleaning is a separate, conservative step that
repairs what it safely can and drops only what cannot describe a real object.

Two details that matter more here than in ordinary detection work:

- **Splits are made at the image level.** Splitting annotations puts two boxes
  from one frame on both sides of the split and leaks. If the images carry a
  provenance field, `split_group_key` holds out whole groups instead — the
  competition's train and test imagery come from different institutional sources,
  so a random same-source split flatters the model.
- **Tiny boxes are kept by default.** A 20-pixel box is often a real small
  organism, which is exactly what the high-resolution path exists to catch.

In [ ]:
"""
Stage 1: clean the supervision before touching the network.

COCO annotation auditing, repair, and image-level splitting for
FathomNet-CLEF 2026. Pure stdlib + NumPy so it is testable without torch.
"""


import collections
import copy
import hashlib
import json
from pathlib import Path

import numpy as np

# --------------------------------------------------------------------------- #
# Loading
# --------------------------------------------------------------------------- #

#: The official repo ships `dataset_train.json` / `dataset_test.json`, but
#: mirrors and older write-ups use the reversed spelling. Accept both.
TRAIN_JSON_NAMES = ("dataset_train.json", "train_dataset.json")
TEST_JSON_NAMES = ("dataset_test.json", "test_dataset.json")


def find_dataset_json(search_roots, names) -> Path | None:
    """
    Locate the first matching annotation file under any of `search_roots`.

    Kaggle mounts competition data at an unpredictable depth under
    /kaggle/input, so searching beats hardcoding a path.
    """
    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue
        for name in names:
            direct = root / name
            if direct.is_file():
                return direct
        for name in names:
            hit = next(iter(sorted(root.rglob(name))), None)
            if hit is not None:
                return hit
    return None


def load_coco(path) -> dict:
    with open(path, "r", encoding="utf-8") as fh:
        coco = json.load(fh)
    for key in ("images", "categories"):
        if key not in coco:
            raise ValueError(f"{path}: missing required COCO key {key!r}")
    coco.setdefault("annotations", [])
    return coco


# --------------------------------------------------------------------------- #
# Audit
# --------------------------------------------------------------------------- #


def audit_coco(coco: dict, min_box_side: float = 2.0, tiny_area: float = 32.0) -> dict:
    """
    Report everything wrong (or merely suspicious) about a COCO file.

    This is read-only and never mutates `coco`. It is the first thing to run on
    the challenge data, because in a positive-unlabeled setting an annotation
    bug and a deliberately-missing label look identical downstream, and only one
    of the two is yours to fix.

    Checked:
      - annotations referencing a missing image_id or category_id
      - non-finite, negative-extent, or zero-area boxes
      - boxes extending outside the declared image dimensions
      - exact duplicate annotations (same image, category, box)
      - near-duplicate annotations (same image+category, IoU >= 0.95)
      - degenerately small boxes
      - images carrying no annotations at all
      - per-category instance counts (imbalance)
      - missing/invalid image width/height
    """

    images = {im["id"]: im for im in coco["images"]}
    cat_ids = {c["id"] for c in coco["categories"]}
    anns = coco["annotations"]

    report: dict = {
        "n_images": len(images),
        "n_annotations": len(anns),
        "n_categories": len(cat_ids),
        "orphan_image_id": [],
        "unknown_category_id": [],
        "non_finite_bbox": [],
        "non_positive_bbox": [],
        "out_of_bounds_bbox": [],
        "tiny_bbox": [],
        "exact_duplicates": [],
        "near_duplicates": [],
        "bad_image_dims": [],
        "images_without_annotations": [],
    }

    for im_id, im in images.items():
        w, h = im.get("width"), im.get("height")
        if not isinstance(w, (int, float)) or not isinstance(h, (int, float)) or w <= 0 or h <= 0:
            report["bad_image_dims"].append(im_id)

    seen: set[tuple] = set()
    by_image_cat: dict[tuple, list] = collections.defaultdict(list)

    for ann in anns:
        aid = ann.get("id")
        im_id = ann.get("image_id")
        cid = ann.get("category_id")

        if im_id not in images:
            report["orphan_image_id"].append(aid)
            continue
        if cid not in cat_ids:
            report["unknown_category_id"].append(aid)

        bbox = ann.get("bbox", None)
        if bbox is None or len(bbox) != 4:
            report["non_finite_bbox"].append(aid)
            continue

        arr = np.asarray(bbox, dtype=np.float64)
        if not np.all(np.isfinite(arr)):
            report["non_finite_bbox"].append(aid)
            continue

        x, y, w, h = arr
        if w <= 0 or h <= 0:
            report["non_positive_bbox"].append(aid)
            continue
        if w < min_box_side or h < min_box_side or (w * h) < tiny_area:
            report["tiny_bbox"].append(aid)

        im = images[im_id]
        iw, ih = im.get("width") or 0, im.get("height") or 0
        if iw > 0 and ih > 0:
            # Allow 1px of float slop before calling it out of bounds.
            if x < -1.0 or y < -1.0 or (x + w) > iw + 1.0 or (y + h) > ih + 1.0:
                report["out_of_bounds_bbox"].append(aid)

        key = (im_id, cid, round(float(x), 3), round(float(y), 3),
               round(float(w), 3), round(float(h), 3))
        if key in seen:
            report["exact_duplicates"].append(aid)
        else:
            seen.add(key)
            by_image_cat[(im_id, cid)].append(ann)

    # Near-duplicates: same image and class, IoU >= 0.95.
    for (im_id, cid), group in by_image_cat.items():
        if len(group) < 2:
            continue
        boxes = xywh_to_xyxy(np.array([g["bbox"] for g in group], dtype=np.float64))
        ious = iou_matrix(boxes, boxes)
        np.fill_diagonal(ious, 0.0)
        ii, jj = np.where(ious >= 0.95)
        for i, j in zip(ii, jj):
            if i < j:
                report["near_duplicates"].append((group[i].get("id"), group[j].get("id")))

    annotated = {a.get("image_id") for a in anns}
    report["images_without_annotations"] = sorted(
        im_id for im_id in images if im_id not in annotated
    )

    counts = collections.Counter(
        a.get("category_id") for a in anns if a.get("image_id") in images
    )
    cat_names = {c["id"]: c.get("name", str(c["id"])) for c in coco["categories"]}
    report["category_counts"] = {
        cat_names.get(cid, str(cid)): n for cid, n in counts.most_common()
    }
    report["categories_with_no_instances"] = sorted(
        cat_names[cid] for cid in cat_ids if counts.get(cid, 0) == 0
    )

    # Instances per image, which is the headline PU statistic: a mean near 1.0
    # on natural underwater scenes is itself evidence of incomplete labelling.
    per_image = collections.Counter(
        a.get("image_id") for a in anns if a.get("image_id") in images
    )
    vals = np.array([per_image.get(i, 0) for i in images], dtype=np.float64)
    report["instances_per_image"] = {
        "mean": float(vals.mean()) if len(vals) else 0.0,
        "median": float(np.median(vals)) if len(vals) else 0.0,
        "max": int(vals.max()) if len(vals) else 0,
        "n_zero": int((vals == 0).sum()),
    }
    return report


def clean_coco(
    coco: dict,
    drop_tiny: bool = False,
    min_box_side: float = 2.0,
    tiny_area: float = 32.0,
    clip_to_image: bool = True,
) -> tuple[dict, dict]:
    """
    Produce a repaired deep copy of `coco` plus a summary of what changed.

    Deliberately conservative. Annotations are dropped only when they cannot
    describe a real object (orphaned, malformed, zero-area, exact duplicate);
    out-of-bounds boxes are clipped rather than discarded, since a box running
    past the frame edge is usually a real organism annotated loosely.

    `drop_tiny` is off by default: in this dataset a 20-pixel box is often a
    genuine small organism, which is exactly what the high-resolution path
    exists to catch. Turn it on only as a deliberate ablation.
    """
    out = copy.deepcopy(coco)
    images = {im["id"]: im for im in out["images"]}
    cat_ids = {c["id"] for c in out["categories"]}

    kept, summary = [], collections.Counter()
    seen: set[tuple] = set()

    for ann in out["annotations"]:
        im_id, cid = ann.get("image_id"), ann.get("category_id")

        if im_id not in images:
            summary["dropped_orphan"] += 1
            continue
        if cid not in cat_ids:
            summary["dropped_unknown_category"] += 1
            continue

        bbox = ann.get("bbox")
        if bbox is None or len(bbox) != 4:
            summary["dropped_malformed"] += 1
            continue
        arr = np.asarray(bbox, dtype=np.float64)
        if not np.all(np.isfinite(arr)):
            summary["dropped_malformed"] += 1
            continue

        x, y, w, h = (float(v) for v in arr)
        if w <= 0 or h <= 0:
            summary["dropped_non_positive"] += 1
            continue

        if clip_to_image:
            iw, ih = images[im_id].get("width") or 0, images[im_id].get("height") or 0
            if iw > 0 and ih > 0:
                x1, y1 = max(0.0, x), max(0.0, y)
                x2, y2 = min(float(iw), x + w), min(float(ih), y + h)
                if (x2 - x1) <= 0 or (y2 - y1) <= 0:
                    summary["dropped_outside_image"] += 1
                    continue
                if (x1, y1, x2, y2) != (x, y, x + w, y + h):
                    summary["clipped"] += 1
                x, y, w, h = x1, y1, x2 - x1, y2 - y1

        if drop_tiny and (w < min_box_side or h < min_box_side or w * h < tiny_area):
            summary["dropped_tiny"] += 1
            continue

        key = (im_id, cid, round(x, 3), round(y, 3), round(w, 3), round(h, 3))
        if key in seen:
            summary["dropped_duplicate"] += 1
            continue
        seen.add(key)

        ann["bbox"] = [x, y, w, h]
        ann["area"] = w * h
        ann.setdefault("iscrowd", 0)
        kept.append(ann)

    out["annotations"] = kept
    summary["kept"] = len(kept)
    summary["input"] = len(coco["annotations"])
    return out, dict(summary)


# --------------------------------------------------------------------------- #
# Splitting
# --------------------------------------------------------------------------- #


def _stable_hash(value, salt: str) -> float:
    """Deterministic hash in [0, 1), stable across processes and Python runs."""
    digest = hashlib.sha256(f"{salt}:{value}".encode("utf-8")).hexdigest()
    return int(digest[:16], 16) / float(1 << 64)


def image_level_split(
    coco: dict,
    val_fraction: float = 0.15,
    salt: str = "fathomnet-pu-v1",
    group_key: str | None = None,
) -> tuple[list, list]:
    """
    Split at the IMAGE level, never the annotation level.

    Splitting annotations puts two boxes from the same frame on both sides of the
    split and leaks. Returns (train_image_ids, val_image_ids).

    `group_key` optionally holds out whole groups instead of individual images --
    pass e.g. "source" or a deployment/dive field if the annotations carry one.
    The competition's train and test imagery come from different institutional
    sources, so a random same-source split flatters the model; grouping is the
    closer proxy for the real evaluation.
    """
    if not (0.0 <= val_fraction < 1.0):
        raise ValueError("val_fraction must be in [0, 1)")

    if group_key:
        groups: dict = collections.defaultdict(list)
        for im in coco["images"]:
            groups[im.get(group_key, "__missing__")].append(im["id"])
        train, val = [], []
        for gname, ids in sorted(groups.items(), key=lambda kv: str(kv[0])):
            (val if _stable_hash(gname, salt) < val_fraction else train).extend(ids)
        return sorted(train), sorted(val)

    train, val = [], []
    for im in coco["images"]:
        (val if _stable_hash(im["id"], salt) < val_fraction else train).append(im["id"])
    return sorted(train), sorted(val)


def subset_coco(coco: dict, image_ids) -> dict:
    """Restrict a COCO dict to `image_ids`, keeping categories intact."""
    wanted = set(image_ids)
    out = {
        "images": [im for im in coco["images"] if im["id"] in wanted],
        "annotations": [a for a in coco["annotations"] if a.get("image_id") in wanted],
        "categories": copy.deepcopy(coco["categories"]),
    }
    for key in ("info", "licenses"):
        if key in coco:
            out[key] = copy.deepcopy(coco[key])
    return out


def merge_pseudo_labels(
    coco: dict,
    pseudo: dict[int, dict],
    mark_key: str = "is_pseudo",
) -> tuple[dict, int]:
    """
    Fold harvested pseudo-labels into a COCO dict as new annotations.

    `pseudo` maps image_id -> {"boxes": (N,4) xyxy, "labels": (N,), "scores": (N,)}.
    Each added annotation is tagged with `mark_key=True` and keeps its originating
    score, so you can down-weight pseudo-labels in the loss, or strip them again
    for an ablation, without re-running the harvest.
    """

    out = copy.deepcopy(coco)
    next_id = max((a.get("id", 0) for a in out["annotations"]), default=0) + 1
    valid_images = {im["id"] for im in out["images"]}
    added = 0

    for im_id, payload in pseudo.items():
        if im_id not in valid_images:
            continue
        boxes = np.asarray(payload["boxes"], dtype=np.float64).reshape(-1, 4)
        labels = np.asarray(payload["labels"]).reshape(-1)
        scores = np.asarray(payload.get("scores", np.ones(len(boxes)))).reshape(-1)
        if len(boxes) == 0:
            continue
        for bb, lab, sc in zip(xyxy_to_xywh(boxes), labels, scores):
            w, h = float(bb[2]), float(bb[3])
            if w <= 0 or h <= 0:
                continue
            out["annotations"].append(
                {
                    "id": next_id,
                    "image_id": im_id,
                    "category_id": int(lab),
                    "bbox": [float(bb[0]), float(bb[1]), w, h],
                    "area": w * h,
                    "iscrowd": 0,
                    mark_key: True,
                    "score": float(sc),
                }
            )
            next_id += 1
            added += 1
    return out, added


# --------------------------------------------------------------------------- #
# Submission
# --------------------------------------------------------------------------- #

#: Exact column order required by the FathomNet-CLEF 2026 submission format.
SUBMISSION_COLUMNS = [
    "annotation_id",
    "image_id",
    "category_id",
    "bbox_x",
    "bbox_y",
    "bbox_width",
    "bbox_height",
    "score",
]


def build_submission(predictions: dict[int, dict], start_id: int = 1):
    """
    Turn per-image predictions into the competition's submission rows.

    `predictions` maps image_id -> {"boxes": (N,4) xyxy in ORIGINAL frame
    coordinates, "scores": (N,), "labels": (N,)}.

    Returns a list of dicts in SUBMISSION_COLUMNS order. Boxes are emitted as
    COCO xywh because that is what the schema's bbox_x/bbox_y/bbox_width/
    bbox_height fields mean -- emitting xyxy here silently destroys the score.
    """

    rows = []
    aid = start_id
    for im_id in sorted(predictions):
        p = predictions[im_id]
        boxes = np.asarray(p["boxes"], dtype=np.float64).reshape(-1, 4)
        scores = np.asarray(p["scores"], dtype=np.float64).reshape(-1)
        labels = np.asarray(p["labels"]).reshape(-1)
        if len(boxes) == 0:
            continue
        for bb, sc, lab in zip(xyxy_to_xywh(boxes), scores, labels):
            rows.append(
                {
                    "annotation_id": aid,
                    "image_id": int(im_id),
                    "category_id": int(lab),
                    "bbox_x": round(float(bb[0]), 2),
                    "bbox_y": round(float(bb[1]), 2),
                    "bbox_width": round(float(bb[2]), 2),
                    "bbox_height": round(float(bb[3]), 2),
                    "score": round(float(np.clip(sc, 0.0, 1.0)), 6),
                }
            )
            aid += 1
    return rows

## 5. Stage 4 — the PU-aware background loss

The heart of the pipeline. A simplified detector objective:

$$L_{\text{total}} = L_{\text{positive}} + \lambda_{\text{box}} L_{\text{box}} + w_{bg}(q)\, L_{\text{background}}$$

Ordinary training sets $w_{bg} \equiv 1$: every unmatched query is background, at
full penalty. The PU version makes that weight depend on how much object evidence
the query carries:

$$w_{bg}(q) = w_{\max} - (w_{\max} - w_{\min})\,\sigma\!\left(\frac{q - \tau}{T}\right)$$

- unmatched **+ weak** object evidence → probably background → full penalty
- unmatched **+ strong** object evidence → possibly a missing annotation → reduced penalty

This is not a claim that confident unmatched predictions are correct. It is the
weaker and more defensible statement: *"I am less certain that this region is negative."*

**`pu_w_min` must stay above zero.** Remove the background penalty entirely and
false positives explode — rocks, coral texture, equipment, sediment and lighting
artifacts all become organisms. The weight buys recall on unlabeled organisms and
pays in precision, and the useful settings are strictly interior:

```
large background weight  ->  fewer false positives, more missed organisms
small background weight  ->  better recall, potentially many false positives
```

Two deliberate implementation choices:

1. **The objectness used for gating is detached.** Otherwise the model could
   lower its own background penalty by becoming more confident — it would learn
   to game the supervision instead of the task.
2. **The criterion is standalone rather than a patch over the HuggingFace loss.**
   HF's internal loss classes move between versions, and a silent API drift there
   would produce a model that trains happily with no PU behaviour at all.

In [ ]:
"""
Stage 4: PU-aware detection criterion (PyTorch).

Implements the blog's objective

    L_total = L_positive + lambda_box * L_box + w_bg(q) * L_background

for an RT-DETR-style head, i.e. sigmoid/focal classification over `num_classes`
with NO explicit background class -- "background" means every class logit is low.

Why a standalone criterion instead of patching the HuggingFace loss:
  * the PU modification is the whole point of the pipeline, so it should be
    readable and testable rather than buried in a monkeypatch;
  * HF's internal loss classes move between versions, and a silent API drift
    here would produce a model that trains without the PU behaviour at all.

It consumes only `logits` and `pred_boxes`, so it works with any DETR-family
model that exposes those two tensors.
"""


from dataclasses import dataclass, field

import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.optimize import linear_sum_assignment


# --------------------------------------------------------------------------- #
# Box helpers (cxcywh normalised <-> xyxy normalised)
# --------------------------------------------------------------------------- #


def cxcywh_to_xyxy(boxes: torch.Tensor) -> torch.Tensor:
    cx, cy, w, h = boxes.unbind(-1)
    return torch.stack(
        [cx - 0.5 * w, cy - 0.5 * h, cx + 0.5 * w, cy + 0.5 * h], dim=-1
    )


def xyxy_to_cxcywh(boxes: torch.Tensor) -> torch.Tensor:
    x1, y1, x2, y2 = boxes.unbind(-1)
    return torch.stack(
        [(x1 + x2) * 0.5, (y1 + y2) * 0.5, (x2 - x1), (y2 - y1)], dim=-1
    )


def box_iou_xyxy(a: torch.Tensor, b: torch.Tensor):
    """Pairwise IoU and union for xyxy boxes. Returns (iou, union)."""
    area_a = (a[:, 2] - a[:, 0]).clamp(min=0) * (a[:, 3] - a[:, 1]).clamp(min=0)
    area_b = (b[:, 2] - b[:, 0]).clamp(min=0) * (b[:, 3] - b[:, 1]).clamp(min=0)

    lt = torch.max(a[:, None, :2], b[None, :, :2])
    rb = torch.min(a[:, None, 2:], b[None, :, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[..., 0] * wh[..., 1]

    union = area_a[:, None] + area_b[None, :] - inter
    iou = inter / union.clamp(min=1e-7)
    return iou, union


def generalized_box_iou(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """GIoU for xyxy boxes. Implemented here to avoid torchvision version drift."""
    iou, union = box_iou_xyxy(a, b)
    lt = torch.min(a[:, None, :2], b[None, :, :2])
    rb = torch.max(a[:, None, 2:], b[None, :, 2:])
    wh = (rb - lt).clamp(min=0)
    enclosing = (wh[..., 0] * wh[..., 1]).clamp(min=1e-7)
    return iou - (enclosing - union) / enclosing


# --------------------------------------------------------------------------- #
# The PU background weight
# --------------------------------------------------------------------------- #


def pu_background_weight_torch(
    objectness: torch.Tensor,
    w_min: float = 0.25,
    w_max: float = 1.0,
    tau: float = 0.5,
    temperature: float = 0.1,
) -> torch.Tensor:
    """
    Torch twin of pu_ops.pu_background_weight_torch -- see that docstring for the idea.

        w(q) = w_max - (w_max - w_min) * sigmoid((q - tau) / temperature)

    An unmatched query with weak object evidence keeps the full background
    penalty. One with strong object evidence has its penalty softened, because it
    may be a missing annotation rather than background.

    Returned detached-safe: the weight is a *supervision* decision, not something
    the model should be able to game by inflating its own confidence, so callers
    pass a detached objectness in.
    """
    if not (0.0 <= w_min <= w_max):
        raise ValueError("require 0 <= w_min <= w_max")
    if temperature <= 0:
        raise ValueError("temperature must be > 0")
    gate = torch.sigmoid((objectness - tau) / temperature)
    return w_max - (w_max - w_min) * gate


def sigmoid_focal_loss(
    logits: torch.Tensor,
    targets: torch.Tensor,
    alpha: float = 0.25,
    gamma: float = 2.0,
    reduction: str = "none",
) -> torch.Tensor:
    """Sigmoid focal loss (Lin et al. 2017), elementwise by default."""
    prob = logits.sigmoid()
    ce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p_t = prob * targets + (1 - prob) * (1 - targets)
    loss = ce * ((1.0 - p_t) ** gamma)
    if alpha >= 0:
        loss = loss * (alpha * targets + (1 - alpha) * (1 - targets))
    if reduction == "sum":
        return loss.sum()
    if reduction == "mean":
        return loss.mean()
    return loss


# --------------------------------------------------------------------------- #
# Config
# --------------------------------------------------------------------------- #


@dataclass
class PULossConfig:
    num_classes: int = 32

    # Matching costs
    cost_class: float = 2.0
    cost_bbox: float = 5.0
    cost_giou: float = 2.0

    # Loss weights
    w_class: float = 1.0
    w_bbox: float = 5.0
    w_giou: float = 2.0

    # Focal loss
    focal_alpha: float = 0.25
    focal_gamma: float = 2.0

    # --- the PU knobs -----------------------------------------------------
    #: Background weight for an unmatched query with STRONG object evidence.
    #: Must stay > 0. At 0 the model receives no background supervision on its
    #: confident mistakes and false positives explode.
    pu_w_min: float = 0.25
    #: Background weight for an unmatched query with weak object evidence.
    pu_w_max: float = 1.0
    #: Objectness above which a query starts to look like a missing annotation.
    pu_tau: float = 0.5
    #: Gate sharpness.
    pu_temperature: float = 0.1
    #: Set True to disable the PU behaviour entirely (ablation baseline).
    pu_disabled: bool = False

    #: Extra multiplier on the classification/box loss of pseudo-labelled GT
    #: boxes, which are less trustworthy than human annotations.
    pseudo_label_weight: float = 0.5

    def background_kwargs(self) -> dict:
        if self.pu_disabled:
            return dict(w_min=self.pu_w_max, w_max=self.pu_w_max,
                        tau=self.pu_tau, temperature=self.pu_temperature)
        return dict(w_min=self.pu_w_min, w_max=self.pu_w_max,
                    tau=self.pu_tau, temperature=self.pu_temperature)


# --------------------------------------------------------------------------- #
# Criterion
# --------------------------------------------------------------------------- #


class PUDetectionCriterion(nn.Module):
    """
    Hungarian-matched detection loss with PU-aware background supervision.

    Targets are a list (length = batch) of dicts:
        {"labels":    LongTensor (n,)            class indices in [0, num_classes)
         "boxes":     FloatTensor (n, 4)         cxcywh, NORMALISED to [0, 1]
         "is_pseudo": BoolTensor  (n,)  optional per-box pseudo-label flag}

    Outputs are a dict:
        {"logits":     FloatTensor (B, Q, num_classes)
         "pred_boxes": FloatTensor (B, Q, 4)     cxcywh, normalised}
    """

    def __init__(self, config: PULossConfig):
        super().__init__()
        self.cfg = config

    # ---------------------------------------------------------------- #
    @torch.no_grad()
    def match(self, outputs: dict, targets: list[dict]) -> list[tuple]:
        """Hungarian assignment between queries and ground-truth boxes."""
        cfg = self.cfg
        logits, pred_boxes = outputs["logits"], outputs["pred_boxes"]
        bs, num_queries = logits.shape[:2]
        indices = []

        for b in range(bs):
            tgt = targets[b]
            n_tgt = int(tgt["labels"].numel())
            if n_tgt == 0:
                indices.append(
                    (
                        torch.zeros(0, dtype=torch.long, device=logits.device),
                        torch.zeros(0, dtype=torch.long, device=logits.device),
                    )
                )
                continue

            prob = logits[b].sigmoid()                      # (Q, C)
            tgt_lab = tgt["labels"].to(logits.device).long()
            tgt_box = tgt["boxes"].to(logits.device).float()

            # Focal-style classification cost, as in Deformable DETR.
            neg = (1 - cfg.focal_alpha) * (prob**cfg.focal_gamma) * (
                -(1 - prob + 1e-8).log()
            )
            pos = cfg.focal_alpha * ((1 - prob) ** cfg.focal_gamma) * (
                -(prob + 1e-8).log()
            )
            cost_class = pos[:, tgt_lab] - neg[:, tgt_lab]   # (Q, n_tgt)

            cost_bbox = torch.cdist(pred_boxes[b], tgt_box, p=1)
            cost_giou = -generalized_box_iou(
                cxcywh_to_xyxy(pred_boxes[b]), cxcywh_to_xyxy(tgt_box)
            )

            C = (
                cfg.cost_class * cost_class
                + cfg.cost_bbox * cost_bbox
                + cfg.cost_giou * cost_giou
            )
            C = torch.nan_to_num(C, nan=1e4, posinf=1e4, neginf=-1e4)

            qi, ti = linear_sum_assignment(C.detach().cpu().numpy())
            indices.append(
                (
                    torch.as_tensor(qi, dtype=torch.long, device=logits.device),
                    torch.as_tensor(ti, dtype=torch.long, device=logits.device),
                )
            )
        return indices

    # ---------------------------------------------------------------- #
    def forward(self, outputs: dict, targets: list[dict]) -> dict:
        cfg = self.cfg
        logits, pred_boxes = outputs["logits"], outputs["pred_boxes"]
        bs, num_queries, num_classes = logits.shape
        device = logits.device

        if num_classes != cfg.num_classes:
            raise ValueError(
                f"model predicts {num_classes} classes but config says {cfg.num_classes}"
            )

        indices = self.match(outputs, targets)

        # Build the dense classification target, plus a per-query mask of which
        # queries were matched and a per-query weight for box/class terms.
        cls_target = torch.zeros_like(logits)
        matched_mask = torch.zeros((bs, num_queries), dtype=torch.bool, device=device)
        pos_weight = torch.ones((bs, num_queries), device=device)

        box_pred_list, box_tgt_list, box_w_list = [], [], []

        for b, (qi, ti) in enumerate(indices):
            if qi.numel() == 0:
                continue
            tgt = targets[b]
            lab = tgt["labels"].to(device).long()[ti]
            box = tgt["boxes"].to(device).float()[ti]

            cls_target[b, qi, lab] = 1.0
            matched_mask[b, qi] = True

            if "is_pseudo" in tgt and tgt["is_pseudo"] is not None:
                is_p = tgt["is_pseudo"].to(device).bool()[ti]
                w = torch.where(
                    is_p,
                    torch.full_like(box[:, 0], cfg.pseudo_label_weight),
                    torch.ones_like(box[:, 0]),
                )
            else:
                w = torch.ones_like(box[:, 0])

            pos_weight[b, qi] = w
            box_pred_list.append(pred_boxes[b, qi])
            box_tgt_list.append(box)
            box_w_list.append(w)

        # ---- classification -------------------------------------------------
        cls_loss_el = sigmoid_focal_loss(
            logits, cls_target, alpha=cfg.focal_alpha, gamma=cfg.focal_gamma
        )                                                     # (B, Q, C)
        cls_loss_q = cls_loss_el.sum(dim=-1)                  # (B, Q)

        # Objectness proxy for an unmatched query: its strongest class score.
        # Detached, so softening the penalty cannot be gamed by the model simply
        # becoming more confident.
        with torch.no_grad():
            objectness = logits.sigmoid().max(dim=-1).values  # (B, Q)
            bg_w = pu_background_weight_torch(objectness, **cfg.background_kwargs())
            bg_w = torch.where(
                matched_mask, torch.ones_like(bg_w), bg_w
            )                                                 # matched rows unaffected

        pos_term = (cls_loss_q * pos_weight * matched_mask).sum()
        bg_term = (cls_loss_q * bg_w * (~matched_mask)).sum()

        # Normalise by the number of matched boxes, the DETR convention.
        num_boxes = max(int(matched_mask.sum().item()), 1)
        loss_class = cfg.w_class * (pos_term + bg_term) / num_boxes

        # ---- boxes ----------------------------------------------------------
        if box_pred_list:
            bp = torch.cat(box_pred_list, dim=0)
            bt = torch.cat(box_tgt_list, dim=0)
            bw = torch.cat(box_w_list, dim=0)

            l1 = (F.l1_loss(bp, bt, reduction="none").sum(dim=-1) * bw).sum() / num_boxes
            giou_mat = generalized_box_iou(cxcywh_to_xyxy(bp), cxcywh_to_xyxy(bt))
            giou = ((1.0 - giou_mat.diag()) * bw).sum() / num_boxes
        else:
            l1 = logits.sum() * 0.0
            giou = logits.sum() * 0.0

        loss_bbox = cfg.w_bbox * l1
        loss_giou = cfg.w_giou * giou
        total = loss_class + loss_bbox + loss_giou

        return {
            "loss": total,
            "loss_class": loss_class.detach(),
            "loss_bbox": loss_bbox.detach(),
            "loss_giou": loss_giou.detach(),
            # Diagnostics worth logging every epoch:
            "num_matched": torch.tensor(float(num_boxes), device=device),
            "mean_bg_weight": bg_w[~matched_mask].mean().detach()
            if (~matched_mask).any()
            else torch.tensor(0.0, device=device),
            "n_softened": (
                (bg_w < cfg.pu_w_max - 1e-6) & (~matched_mask)
            ).sum().detach(),
        }

### 5b. Self-test for the PU criterion

Runs on CPU in a few seconds. It checks the properties that actually matter — in
particular that `pu_disabled=True` reproduces the standard loss exactly, and that
softening the background weight only ever affects *unmatched* queries.

In [ ]:
def selftest_pu_criterion(verbose=True):
    import torch
    torch.manual_seed(0)

    B, Q, C, N = 2, 24, 6, 3
    logits = torch.randn(B, Q, C, requires_grad=True)
    boxes = torch.rand(B, Q, 4) * 0.5 + 0.25
    boxes = boxes.clone().requires_grad_(True)
    outputs = {"logits": logits, "pred_boxes": boxes}
    targets = [
        {"labels": torch.randint(0, C, (N,)), "boxes": torch.rand(N, 4) * 0.4 + 0.3}
        for _ in range(B)
    ]

    results, failures = [], []

    def ck(name, cond, detail=""):
        results.append((name, bool(cond)))
        if not cond:
            failures.append(f"{name} {detail}")
        if verbose:
            print(f"  {'PASS' if cond else 'FAIL'}  {name}  {detail if not cond else ''}")

    base = PULossConfig(num_classes=C)
    crit = PUDetectionCriterion(base)
    out = crit(outputs, targets)

    ck("loss is finite", torch.isfinite(out["loss"]).item(), f"{out['loss']}")
    ck("loss is positive", out["loss"].item() > 0, f"{out['loss'].item()}")

    out["loss"].backward()
    ck("gradient reaches logits", logits.grad is not None and torch.isfinite(logits.grad).all().item())
    ck("gradient reaches boxes", boxes.grad is not None and torch.isfinite(boxes.grad).all().item())
    ck("gradient is non-zero", logits.grad.abs().sum().item() > 0)

    # Matching must be one-to-one and cover every target.
    idx = crit.match(outputs, targets)
    ck("matched count == n targets", all(len(q) == N for q, t in idx))
    ck("query indices unique", all(len(set(q.tolist())) == len(q) for q, t in idx))
    ck("target indices are a permutation", all(sorted(t.tolist()) == list(range(N)) for q, t in idx))

    # pu_disabled must be numerically identical to w_min == w_max.
    a = PUDetectionCriterion(PULossConfig(num_classes=C, pu_disabled=True))(outputs, targets)
    b = PUDetectionCriterion(
        PULossConfig(num_classes=C, pu_w_min=1.0, pu_w_max=1.0)
    )(outputs, targets)
    ck("pu_disabled == uniform background weight",
       abs(a["loss"].item() - b["loss"].item()) < 1e-6,
       f"{a['loss'].item()} vs {b['loss'].item()}")
    ck("pu_disabled softens nothing", a["n_softened"].item() == 0, f"{a['n_softened'].item()}")

    # Lowering w_min must lower total loss (less background penalty) and must
    # leave the box terms untouched, since it only touches unmatched queries.
    hi = PUDetectionCriterion(PULossConfig(num_classes=C, pu_w_min=1.0))(outputs, targets)
    lo = PUDetectionCriterion(PULossConfig(num_classes=C, pu_w_min=0.1))(outputs, targets)
    ck("lower w_min lowers total loss", lo["loss"].item() < hi["loss"].item(),
       f"{lo['loss'].item()} vs {hi['loss'].item()}")
    ck("w_min does not touch box loss",
       abs(lo["loss_bbox"].item() - hi["loss_bbox"].item()) < 1e-6)
    ck("w_min does not touch giou loss",
       abs(lo["loss_giou"].item() - hi["loss_giou"].item()) < 1e-6)
    ck("mean background weight drops with w_min",
       lo["mean_bg_weight"].item() < hi["mean_bg_weight"].item(),
       f"{lo['mean_bg_weight'].item()} vs {hi['mean_bg_weight'].item()}")

    # Weight bounds, and agreement with the NumPy twin from section 3. The two
    # implementations are independent, so this is a real cross-check.
    w = pu_background_weight_torch(torch.linspace(0, 1, 11), w_min=0.25, w_max=1.0,
                                   tau=0.5, temperature=0.1)
    ck("torch weight within bounds", bool((w >= 0.25).all() and (w <= 1.0).all()))
    ck("torch weight strictly decreasing", bool((w.diff() < 0).all()))
    ck("torch weight matches numpy twin",
       np.allclose(w.numpy(), pu_background_weight(np.linspace(0, 1, 11),
                   w_min=0.25, w_max=1.0, tau=0.5, temperature=0.1), atol=1e-6),
       "torch and numpy PU weights disagree")

    # Empty-target images must not crash or produce NaNs.
    empty = [{"labels": torch.zeros(0, dtype=torch.long), "boxes": torch.zeros(0, 4)}
             for _ in range(B)]
    oe = PUDetectionCriterion(base)(outputs, empty)
    ck("empty targets handled", torch.isfinite(oe["loss"]).item(), f"{oe['loss']}")

    # Pseudo-label down-weighting must reduce the positive contribution.
    tp = [dict(t, is_pseudo=torch.ones(N, dtype=torch.bool)) for t in targets]
    p_full = PUDetectionCriterion(PULossConfig(num_classes=C, pseudo_label_weight=1.0))(outputs, tp)
    p_half = PUDetectionCriterion(PULossConfig(num_classes=C, pseudo_label_weight=0.25))(outputs, tp)
    ck("pseudo-label weight reduces box loss",
       p_half["loss_bbox"].item() < p_full["loss_bbox"].item(),
       f"{p_half['loss_bbox'].item()} vs {p_full['loss_bbox'].item()}")

    # Mismatched class count must fail loudly, not silently.
    try:
        PUDetectionCriterion(PULossConfig(num_classes=C + 1))(outputs, targets)
        ck("rejects class-count mismatch", False, "no raise")
    except ValueError:
        ck("rejects class-count mismatch", True)

    n_pass = sum(1 for _, ok in results if ok)
    print(f"\n  criterion self-test: {n_pass}/{len(results)} passed")
    if failures:
        raise AssertionError("PU criterion self-test failures:\n  " + "\n  ".join(failures))
    return True

## 6. Loading the challenge data

The official repository ships `dataset_train.json` and `dataset_test.json` in COCO
format, plus a `download.py` — **the imagery is not bundled**, it is fetched from
FathomNet and partner URLs recorded in the annotations. So there are two ways to
get frames, and the resolver below accepts either:

1. an images directory already attached as a Kaggle dataset, or
2. downloading on demand from each image's URL, cached under `work_dir`
   (needs *Settings → Internet → On*).

`CFG.max_images` caps how many frames are used. Leave it small for the first pass.

In [ ]:
train_json_path = find_dataset_json(CFG.input_roots, TRAIN_JSON_NAMES)
test_json_path = find_dataset_json(CFG.input_roots, TEST_JSON_NAMES)

print("train json:", train_json_path)
print("test  json:", test_json_path)

if train_json_path is None:
    print(
        "\n>>> Training annotations not found.\n"
        ">>> Attach the FathomNet-CLEF 2026 competition data (Add Input), or drop\n"
        f">>> dataset_train.json under one of: {list(CFG.input_roots)}\n"
        ">>> The 'verify' stage still runs without it."
    )
    raw_train = None
else:
    raw_train = load_coco(train_json_path)
    print(f"\nimages      : {len(raw_train['images'])}")
    print(f"annotations : {len(raw_train['annotations'])}")
    print(f"categories  : {len(raw_train['categories'])}")

    # Which field holds the image URL varies between COCO exports.
    URL_KEYS = ("coco_url", "url", "flickr_url", "image_url", "file_url")
    sample = raw_train["images"][0]
    print("\nimage record keys:", sorted(sample.keys()))
    found = [k for k in URL_KEYS if sample.get(k)]
    print("url field(s)     :", found or "none - imagery must be attached as a dataset")

In [ ]:
import urllib.request
from concurrent.futures import ThreadPoolExecutor

URL_KEYS = ("coco_url", "url", "flickr_url", "image_url", "file_url")


def image_url(im: dict):
    for k in URL_KEYS:
        v = im.get(k)
        if isinstance(v, str) and v.startswith("http"):
            return v
    return None


def local_image_path(im: dict, cache_dir: Path) -> Path:
    """Stable on-disk name for an image record."""
    name = im.get("file_name") or ""
    if name and not name.startswith("http"):
        stem = Path(name).name
    else:
        url = image_url(im) or str(im["id"])
        stem = hashlib.sha1(url.encode()).hexdigest()[:20] + ".png"
    return cache_dir / stem


def find_attached_image_dir(roots, probe_names, max_probe=40):
    """Look for an already-attached directory containing the expected frames."""
    probe = {Path(n).name for n in probe_names if n}
    if not probe:
        return None
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for d in [root] + [p for p in root.rglob("*") if p.is_dir()][:2000]:
            try:
                names = {p.name for p in list(d.iterdir())[:2000] if p.is_file()}
            except Exception:
                continue
            if names and len(probe & names) >= min(3, len(probe)):
                return d
    return None


def ensure_images(coco, cache_dir, workers=16, verbose=True):
    """
    Make sure every image in `coco` exists on disk; download the missing ones.

    Returns (ok_image_ids, failed_image_ids). Images that cannot be fetched are
    reported rather than silently producing black frames.
    """
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    todo = []
    ok, failed = [], []
    for im in coco["images"]:
        p = local_image_path(im, cache_dir)
        im["_local_path"] = str(p)
        if p.is_file() and p.stat().st_size > 0:
            ok.append(im["id"])
        else:
            u = image_url(im)
            if u:
                todo.append((im["id"], u, p))
            else:
                failed.append(im["id"])

    if todo and verbose:
        print(f"downloading {len(todo)} frames with {workers} workers ...")

    def fetch(job):
        im_id, url, path = job
        for attempt in range(3):
            try:
                req = urllib.request.Request(
                    url, headers={"User-Agent": "fathomnet-pu-notebook/1.0"}
                )
                with urllib.request.urlopen(req, timeout=30) as r:
                    data = r.read()
                if not data:
                    raise IOError("empty response")
                tmp = path.with_suffix(path.suffix + ".part")
                tmp.write_bytes(data)
                tmp.rename(path)
                return im_id, True
            except Exception:
                time.sleep(1.5 * (attempt + 1))
        return im_id, False

    if todo:
        with ThreadPoolExecutor(max_workers=workers) as ex:
            for i, (im_id, good) in enumerate(ex.map(fetch, todo), 1):
                (ok if good else failed).append(im_id)
                if verbose and i % 50 == 0:
                    print(f"  {i}/{len(todo)}")

    if verbose:
        print(f"available: {len(ok)}   unavailable: {len(failed)}")
    return set(ok), set(failed)

## 7. Stage 1 in action — audit, clean, split

The audit is the first real result of the pipeline. The statistic to look at is
**mean instances per image**: on natural underwater scenes a value near 1.0 is
itself evidence of incomplete labelling, and it is the number that should rise
after Stage 3 recovers missing positives.

In [ ]:
AUDIT = None
coco_clean = None
train_ids, val_ids = [], []

if raw_train is not None:
    AUDIT = audit_coco(raw_train)

    print("=" * 62)
    print("STAGE 1 AUDIT")
    print("=" * 62)
    for key in (
        "n_images", "n_annotations", "n_categories",
        "orphan_image_id", "unknown_category_id", "non_finite_bbox",
        "non_positive_bbox", "out_of_bounds_bbox", "tiny_bbox",
        "exact_duplicates", "near_duplicates", "bad_image_dims",
    ):
        v = AUDIT[key]
        print(f"{key:24s}: {v if isinstance(v, int) else len(v)}")
    print(f"{'images w/o annotations':24s}: {len(AUDIT['images_without_annotations'])}")
    print(f"{'empty categories':24s}: {AUDIT['categories_with_no_instances']}")
    print(f"\ninstances per image      : {AUDIT['instances_per_image']}")

    ipi = AUDIT["instances_per_image"]["mean"]
    print(
        f"\n>>> mean {ipi:.2f} instances/image."
        + (
            "  Consistent with heavily incomplete labelling."
            if ipi < 2.0
            else "  Higher than expected - check the PU assumption holds."
        )
    )

    print("\ntop 10 categories by instance count:")
    for name, n in list(AUDIT["category_counts"].items())[:10]:
        print(f"  {name:28s} {n:6d}")
    if AUDIT["category_counts"]:
        counts = list(AUDIT["category_counts"].values())
        print(f"\nimbalance ratio (max/min): {max(counts) / max(min(counts), 1):.1f}x")

In [ ]:
if raw_train is not None:
    coco_clean, clean_summary = clean_coco(
        raw_train, drop_tiny=CFG.drop_tiny_boxes, clip_to_image=True
    )
    print("cleaning summary:")
    for k in sorted(clean_summary):
        print(f"  {k:26s} {clean_summary[k]}")

    # Image-level split. Group-wise if a provenance field is available.
    gk = CFG.split_group_key or None
    if gk and gk not in coco_clean["images"][0]:
        print(f"\n(no '{gk}' field on image records - falling back to per-image split)")
        gk = None
    train_ids, val_ids = image_level_split(
        coco_clean, val_fraction=CFG.val_fraction, salt=CFG.split_salt, group_key=gk
    )
    assert not (set(train_ids) & set(val_ids)), "split leaked"
    print(f"\nsplit: {len(train_ids)} train / {len(val_ids)} val"
          f"  (group_key={gk!r})")

In [ ]:
# Visual check on the two distributions that shape every later decision.
if AUDIT is not None and AUDIT["category_counts"]:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

    names = list(AUDIT["category_counts"])[:20]
    vals = [AUDIT["category_counts"][n] for n in names]
    axes[0].barh(range(len(names)), vals, color="#0f766e")
    axes[0].set_yticks(range(len(names)))
    axes[0].set_yticklabels(names, fontsize=8)
    axes[0].invert_yaxis()
    axes[0].set_title("Instances per category (top 20)")
    axes[0].set_xlabel("instances")

    per_image = collections.Counter(a["image_id"] for a in coco_clean["annotations"])
    vals = [per_image.get(im["id"], 0) for im in coco_clean["images"]]
    axes[1].hist(vals, bins=range(0, max(vals) + 2), color="#0891b2",
                 edgecolor="white", align="left")
    axes[1].axvline(np.mean(vals), color="#b91c1c", ls="--",
                    label=f"mean {np.mean(vals):.2f}")
    axes[1].set_title("Annotated instances per image")
    axes[1].set_xlabel("boxes in image")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

## 8. Stage 2 — a single high-resolution RT-DETR detector

One detector, not an ensemble. The reason for high resolution is concrete: many
marine organisms occupy a small fraction of the frame, and aggressive resizing
erases the shape cues that separate a brittle star from background texture.

Boxes are stored **normalised** `cxcywh`, which makes them invariant to the
resize — the same target tensor is correct at every inference scale, and mapping
detections back to the original frame is a single multiply by `(W, H)`.

In [ ]:
def build_dataset_classes():
    """Defined inside a function so the CPU-only stages never import torch."""
    import torch
    from torch.utils.data import Dataset
    from PIL import Image

    IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    class FathomNetDataset(Dataset):
        """
        COCO-format marine detection dataset.

        Emits normalised cxcywh boxes and contiguous class indices in
        [0, num_classes). `is_pseudo` is carried through so the loss can trust
        human annotations more than harvested ones.
        """

        def __init__(self, coco, image_ids, cat_id_to_index, img_size,
                     train=True, hflip_p=0.5):
            self.img_size = int(img_size)
            self.train = train
            self.hflip_p = hflip_p
            self.cat_id_to_index = dict(cat_id_to_index)

            keep = set(image_ids)
            self.images = [im for im in coco["images"] if im["id"] in keep]
            self.by_image = collections.defaultdict(list)
            for a in coco["annotations"]:
                if a["image_id"] in keep:
                    self.by_image[a["image_id"]].append(a)

        def __len__(self):
            return len(self.images)

        def _load(self, im):
            path = im.get("_local_path")
            if path and Path(path).is_file():
                return Image.open(path).convert("RGB")
            raise FileNotFoundError(f"image {im['id']} not on disk")

        def __getitem__(self, i):
            im = self.images[i]
            try:
                pil = self._load(im)
            except Exception:
                # A missing frame must not kill an epoch, but it must be visible.
                pil = Image.new("RGB", (self.img_size, self.img_size), (0, 0, 0))
                W, H = pil.size
                return self._pack(pil, [], im["id"], W, H, missing=True)

            W, H = pil.size
            anns = self.by_image.get(im["id"], [])
            return self._pack(pil, anns, im["id"], W, H, missing=False)

        def _pack(self, pil, anns, image_id, W, H, missing):
            boxes, labels, is_pseudo = [], [], []
            for a in anns:
                x, y, w, h = a["bbox"]
                if w <= 0 or h <= 0:
                    continue
                idx = self.cat_id_to_index.get(a["category_id"])
                if idx is None:
                    continue
                # -> normalised cxcywh, invariant to the resize below.
                boxes.append([(x + w / 2) / W, (y + h / 2) / H, w / W, h / H])
                labels.append(idx)
                is_pseudo.append(bool(a.get("is_pseudo", False)))

            pil = pil.resize((self.img_size, self.img_size), Image.BILINEAR)

            if self.train and boxes and random.random() < self.hflip_p:
                pil = pil.transpose(Image.FLIP_LEFT_RIGHT)
                boxes = [[1.0 - b[0], b[1], b[2], b[3]] for b in boxes]

            x = torch.from_numpy(np.asarray(pil, dtype=np.float32).copy())
            x = x.permute(2, 0, 1) / 255.0
            x = (x - IMAGENET_MEAN) / IMAGENET_STD

            t = {
                "labels": torch.as_tensor(labels, dtype=torch.long),
                "boxes": torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4),
                "is_pseudo": torch.as_tensor(is_pseudo, dtype=torch.bool),
                "image_id": int(image_id),
                "orig_size": (int(W), int(H)),
                "missing": bool(missing),
            }
            return x, t

    def collate(batch):
        xs = torch.stack([b[0] for b in batch])
        return xs, [b[1] for b in batch]

    return FathomNetDataset, collate


def build_category_maps(coco):
    """Contiguous 0..K-1 indices, with maps back to the competition's ids."""
    cats = sorted(coco["categories"], key=lambda c: c["id"])
    cat_id_to_index = {c["id"]: i for i, c in enumerate(cats)}
    index_to_cat_id = {i: c["id"] for i, c in enumerate(cats)}
    index_to_name = {i: c.get("name", str(c["id"])) for i, c in enumerate(cats)}
    return cat_id_to_index, index_to_cat_id, index_to_name

In [ ]:
def build_model(num_classes, model_name, device):
    """
    RT-DETR with a fresh classification head sized to the challenge's classes.

    `ignore_mismatched_sizes=True` is what allows the pretrained COCO head to be
    replaced. Requires internet on first call; the weights then sit in the HF cache.
    """
    import torch
    from transformers import RTDetrForObjectDetection

    model = RTDetrForObjectDetection.from_pretrained(
        model_name, num_labels=num_classes, ignore_mismatched_sizes=True
    )
    model.to(device)
    return model


def build_optimizer(model, cfg):
    """
    AdamW with a lower LR on the pretrained backbone.

    Returns (optimizer, max_lrs) so the scheduler's max_lr always lines up with
    the param groups that actually exist -- an empty backbone group (if the
    parameter naming ever changes) would otherwise silently desynchronise them.
    """
    import torch

    backbone, rest = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (backbone if "backbone" in n else rest).append(p)

    groups, max_lrs = [], []
    if rest:
        groups.append({"params": rest, "lr": cfg.lr})
        max_lrs.append(cfg.lr)
    if backbone:
        groups.append({"params": backbone, "lr": cfg.lr_backbone})
        max_lrs.append(cfg.lr_backbone)
    if not groups:
        raise RuntimeError("no trainable parameters found")

    print(f"param groups: head/neck {len(rest)} tensors, backbone {len(backbone)} tensors")
    return torch.optim.AdamW(groups, weight_decay=cfg.weight_decay), max_lrs


# --- AMP compatibility ---------------------------------------------------- #
# torch.cuda.amp.* is deprecated in favour of torch.amp.* from 2.4 onward.
# Kaggle's torch version moves; these shims work either way.

def make_grad_scaler(enabled):
    import torch
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def autocast_ctx(enabled):
    import torch
    try:
        return torch.amp.autocast("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.autocast(enabled=enabled)


def load_checkpoint(path, map_location):
    """torch.load flipped its weights_only default in 2.6; handle both."""
    import torch
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def train_one_run(coco, train_ids, val_ids, cfg, criterion_config,
                  ckpt_name, init_from=None):
    """
    Train the detector and checkpoint every epoch.

    Checkpointing every epoch is not optional on Kaggle: a session that hits the
    wall mid-run otherwise loses the entire GPU allocation it consumed.
    """
    import torch
    from torch.utils.data import DataLoader

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    FathomNetDataset, collate = build_dataset_classes()
    cat_id_to_index, _, _ = build_category_maps(coco)

    ds_tr = FathomNetDataset(coco, train_ids, cat_id_to_index, cfg.img_size, train=True)
    ds_va = FathomNetDataset(coco, val_ids, cat_id_to_index, cfg.img_size, train=False)
    print(f"train images {len(ds_tr)} | val images {len(ds_va)}")

    dl_tr = DataLoader(ds_tr, batch_size=cfg.batch_size, shuffle=True,
                       num_workers=cfg.num_workers, collate_fn=collate,
                       drop_last=True, pin_memory=True)

    model = build_model(cfg.num_classes, cfg.model_name, device)
    if init_from and Path(init_from).is_file():
        model.load_state_dict(load_checkpoint(init_from, device)["model"])
        print(f"initialised from {init_from}")

    criterion = PUDetectionCriterion(criterion_config).to(device)
    optimizer, max_lrs = build_optimizer(model, cfg)
    steps_per_epoch = max(len(dl_tr) // cfg.grad_accum, 1)
    total_steps = max(steps_per_epoch * cfg.epochs, 1)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=max_lrs, total_steps=total_steps, pct_start=0.1
    )
    use_amp = bool(cfg.amp and device.type == "cuda")
    scaler = make_grad_scaler(use_amp)

    ckpt_path = Path(cfg.work_dir) / ckpt_name
    history = []

    for epoch in range(cfg.epochs):
        model.train()
        running = collections.defaultdict(float)
        n_batches = 0
        t0 = time.time()
        optimizer.zero_grad(set_to_none=True)

        for step, (pixel_values, targets) in enumerate(dl_tr):
            pixel_values = pixel_values.to(device, non_blocking=True)

            with autocast_ctx(use_amp):
                out = model(pixel_values=pixel_values)

            # The loss runs outside autocast: the Hungarian matching cost and the
            # focal/GIoU terms are numerically touchy in fp16, and this costs
            # nothing since the backbone forward is where the time goes.
            losses = criterion(
                {"logits": out.logits.float(), "pred_boxes": out.pred_boxes.float()},
                targets,
            )
            loss = losses["loss"] / cfg.grad_accum

            if not torch.isfinite(loss):
                print(f"  !! non-finite loss at step {step}, skipping batch")
                optimizer.zero_grad(set_to_none=True)
                continue

            scaler.scale(loss).backward()

            if (step + 1) % cfg.grad_accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.clip_grad)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                if scheduler.last_epoch < total_steps - 1:
                    scheduler.step()

            for k, v in losses.items():
                running[k] += float(v.detach().item() if hasattr(v, "item") else v)
            n_batches += 1

            if step % 50 == 0:
                print(
                    f"  e{epoch} s{step}/{len(dl_tr)} "
                    f"loss {losses['loss'].item():.3f} "
                    f"cls {losses['loss_class'].item():.3f} "
                    f"bbox {losses['loss_bbox'].item():.3f} "
                    f"matched {int(losses['num_matched'].item())} "
                    f"bg_w {losses['mean_bg_weight'].item():.3f} "
                    f"softened {int(losses['n_softened'].item())}"
                )

        epoch_stats = {k: v / max(n_batches, 1) for k, v in running.items()}
        epoch_stats["epoch"] = epoch
        epoch_stats["seconds"] = round(time.time() - t0, 1)
        history.append(epoch_stats)
        print(f"epoch {epoch} done in {epoch_stats['seconds']}s  "
              f"mean loss {epoch_stats['loss']:.4f}")

        torch.save(
            {
                "model": model.state_dict(),
                "epoch": epoch,
                "cfg": asdict(cfg),
                "criterion_config": asdict(criterion_config),
                "history": history,
            },
            ckpt_path,
        )
        print(f"  checkpoint -> {ckpt_path}")

    return model, history, ckpt_path

## 9. Stage 5 — two-scale inference and Soft-NMS

Two passes instead of a large test-time-augmentation ensemble: the
medium-resolution pass supplies scene context, the high-resolution pass recovers
small organisms. Because the model emits normalised boxes, both passes denormalise
into the original frame directly, and the merge happens in original-frame
coordinates — which is also the frame the metric is computed in.

Merging uses **per-class** Soft-NMS. Cross-class suppression would be wrong here:
a shrimp on a sponge produces two heavily overlapping boxes of different classes
and both are correct. And Soft-NMS rather than hard NMS because dense marine
scenes contain genuinely overlapping organisms — decaying a neighbour's score
degrades gracefully where deleting it loses a real detection.

In [ ]:
def predict_two_scale(model, coco, image_ids, cfg, index_to_cat_id,
                      views_for_consistency=False):
    """
    Run the detector at each scale in cfg.infer_scales and merge.

    Returns image_id -> {"boxes" (xyxy, ORIGINAL frame), "scores", "labels"},
    where labels are competition category_ids. With
    views_for_consistency=True it also returns the per-view detections, which is
    what Stage 3 needs to test whether a candidate survives a transformation.
    """
    import torch
    from PIL import Image

    device = next(model.parameters()).device
    model.eval()

    IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    by_id = {im["id"]: im for im in coco["images"]}
    out_final, out_views = {}, {}

    weights = list(cfg.infer_scale_weights)
    if len(weights) < len(cfg.infer_scales):
        weights += [1.0] * (len(cfg.infer_scales) - len(weights))

    def prep(pil, size):
        x = pil.resize((size, size), Image.BILINEAR)
        x = torch.from_numpy(np.asarray(x, dtype=np.float32).copy())
        x = x.permute(2, 0, 1) / 255.0
        return ((x - IMAGENET_MEAN) / IMAGENET_STD).unsqueeze(0)

    @torch.no_grad()
    def run(pil, size, W, H, flip=False):
        if flip:
            pil = pil.transpose(Image.FLIP_LEFT_RIGHT)
        pv = prep(pil, size).to(device)
        with autocast_ctx(bool(cfg.amp and device.type == "cuda")):
            o = model(pixel_values=pv)
        prob = o.logits[0].sigmoid().float().cpu().numpy()        # (Q, C)
        boxes = o.pred_boxes[0].float().cpu().numpy()             # (Q, 4) cxcywh norm

        # Flatten query x class into independent candidates, as DETR-family
        # post-processing does: one query may legitimately support two classes.
        q_idx, c_idx = np.where(prob >= cfg.score_threshold)
        if len(q_idx) == 0:
            return (np.zeros((0, 4)), np.zeros(0), np.zeros(0, dtype=int))

        scores = prob[q_idx, c_idx]
        b = boxes[q_idx]
        cx, cy, bw, bh = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
        if flip:
            cx = 1.0 - cx
        xyxy = np.stack(
            [(cx - bw / 2) * W, (cy - bh / 2) * H,
             (cx + bw / 2) * W, (cy + bh / 2) * H], axis=1
        )
        return xyxy, scores, c_idx

    for n, im_id in enumerate(image_ids, 1):
        im = by_id[im_id]
        path = im.get("_local_path")
        if not path or not Path(path).is_file():
            continue
        try:
            pil = Image.open(path).convert("RGB")
        except Exception:
            continue
        W, H = pil.size

        passes, views = [], []
        for size, w in zip(cfg.infer_scales, weights):
            xyxy, scores, labels = run(pil, size, W, H)
            passes.append({"boxes": xyxy, "scores": scores, "labels": labels,
                           "size": (W, H), "weight": float(w)})
            views.append({"boxes": xyxy, "scores": scores, "labels": labels})

        if views_for_consistency:
            # One extra flipped view at the largest scale: a texture artifact
            # rarely reappears at the same place and class under a flip.
            xyxy, scores, labels = run(pil, max(cfg.infer_scales), W, H, flip=True)
            views.append({"boxes": xyxy, "scores": scores, "labels": labels})

        merged = merge_multiscale(
            passes,
            original_size=(W, H),
            method=cfg.softnms_method,
            sigma=cfg.softnms_sigma,
            iou_threshold=cfg.softnms_iou_threshold,
            score_threshold=cfg.score_threshold,
        )
        if len(merged["scores"]) > cfg.max_dets_per_image:
            keep = np.argsort(-merged["scores"])[: cfg.max_dets_per_image]
            merged = {k: merged[k][keep] for k in ("boxes", "scores", "labels")}

        # Map contiguous indices back to competition category_ids.
        merged["labels"] = np.array(
            [index_to_cat_id[int(i)] for i in merged["labels"]], dtype=np.int64
        )
        out_final[im_id] = merged
        if views_for_consistency:
            out_views[im_id] = views

        if n % 100 == 0:
            print(f"  inferred {n}/{len(image_ids)}")

    return (out_final, out_views) if views_for_consistency else out_final

## 10. Stage 3 — conservative pseudo-label recovery

Self-training, kept deliberately timid. A pseudo-label earns its place only if it
is more likely to *correct* missing supervision than to *inject* a new error, so
every step in the funnel removes candidates and none adds any:

```
confidence filter  ->  IoU dedup vs GT  ->  cross-view consistency  ->  per-image cap
```

The per-stage survivor counts are the thing to watch. If `after_consistency` is
close to `after_gt_dedup`, the consistency check is not discriminating and the
thresholds need tightening; if it collapses to near zero, nothing is being
recovered and the whole stage is a no-op.

In [ ]:
def harvest_over_dataset(model, coco, image_ids, cfg, index_to_cat_id):
    """Run the Stage 3 funnel across a set of images and aggregate the counts."""
    cat_index_of = {v: k for k, v in index_to_cat_id.items()}

    _, views_by_image = predict_two_scale(
        model, coco, image_ids, cfg, index_to_cat_id, views_for_consistency=True
    )

    gt_by_image = collections.defaultdict(list)
    for a in coco["annotations"]:
        x, y, w, h = a["bbox"]
        gt_by_image[a["image_id"]].append(
            ([x, y, x + w, y + h], cat_index_of.get(a["category_id"], -1))
        )

    harvest, totals = {}, collections.Counter()
    for im_id, views in views_by_image.items():
        gt = gt_by_image.get(im_id, [])
        gt_boxes = np.array([g[0] for g in gt], dtype=float).reshape(-1, 4)
        gt_labels = np.array([g[1] for g in gt], dtype=int)

        res = harvest_pseudo_labels(
            views,
            gt_boxes=gt_boxes,
            gt_labels=gt_labels,
            score_threshold=cfg.pseudo_score_threshold,
            gt_iou_threshold=cfg.pseudo_gt_iou_threshold,
            consistency_iou=cfg.pseudo_consistency_iou,
            min_views=cfg.pseudo_min_views,
            max_per_image=cfg.pseudo_max_per_image,
        )
        for k, v in res["counts"].items():
            totals[k] += v
        if len(res["boxes"]):
            # Back to competition category_ids for merging into the COCO file.
            res["labels"] = np.array(
                [index_to_cat_id[int(i)] for i in res["labels"]], dtype=np.int64
            )
            harvest[im_id] = res

    print("\nStage 3 funnel across", len(views_by_image), "images:")
    for k in ("candidates", "after_confidence", "after_gt_dedup",
              "after_consistency", "kept"):
        print(f"  {k:20s} {totals[k]}")
    if totals["after_gt_dedup"]:
        survival = totals["after_consistency"] / totals["after_gt_dedup"]
        print(f"\n  consistency survival rate: {survival:.1%}")
        if survival > 0.9:
            print("  >>> consistency check is barely filtering - raise consistency_iou")
        elif survival < 0.05:
            print("  >>> almost nothing survives - lower the thresholds or this stage is a no-op")
    return harvest, dict(totals)

## 11. Evaluation — and why one mAP number is not enough

PU learning breaks validation too. If a validation frame contains an unlabeled
fish and the model finds it, the evaluator scores that as a false positive. So a
model that gets **better** at discovering unlabeled organisms can look **worse**
on an incompletely annotated split.

The consequence is practical: mAP computed on the incomplete training split is a
*relative* signal at best, and the diagnostics below matter as much as the number.
A rising count of high-confidence "false positives" alongside stable recall on
confidently-labeled classes is the signature of the PU loss working, not failing.

In [ ]:
def evaluate_map(coco_gt_dict, predictions, index_to_name=None, verbose=True):
    """COCO mAP via pycocotools, plus the PU diagnostics worth reading beside it."""
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    import tempfile

    gt = {
        "images": coco_gt_dict["images"],
        "annotations": [
            dict(a, iscrowd=a.get("iscrowd", 0), area=a.get("area", a["bbox"][2] * a["bbox"][3]))
            for a in coco_gt_dict["annotations"]
        ],
        "categories": coco_gt_dict["categories"],
    }
    for i, a in enumerate(gt["annotations"], 1):
        a.setdefault("id", i)

    dets = []
    for im_id, p in predictions.items():
        for bb, sc, lab in zip(xyxy_to_xywh(p["boxes"]), p["scores"], p["labels"]):
            dets.append({
                "image_id": int(im_id), "category_id": int(lab),
                "bbox": [float(v) for v in bb], "score": float(sc),
            })

    if not dets:
        print("no detections - nothing to evaluate")
        return {"mAP": 0.0, "mAP50": 0.0, "n_dets": 0}

    with tempfile.TemporaryDirectory() as td:
        gt_path = Path(td) / "gt.json"
        dt_path = Path(td) / "dt.json"
        gt_path.write_text(json.dumps(gt))
        dt_path.write_text(json.dumps(dets))

        coco_gt = COCO(str(gt_path))
        coco_dt = coco_gt.loadRes(str(dt_path))
        e = COCOeval(coco_gt, coco_dt, "bbox")
        e.params.imgIds = sorted(predictions)
        e.evaluate(); e.accumulate()
        if verbose:
            e.summarize()
        else:
            import io, contextlib
            with contextlib.redirect_stdout(io.StringIO()):
                e.summarize()

    stats = {
        "mAP": float(e.stats[0]), "mAP50": float(e.stats[1]), "mAP75": float(e.stats[2]),
        "mAP_small": float(e.stats[3]), "mAP_medium": float(e.stats[4]),
        "mAP_large": float(e.stats[5]), "n_dets": len(dets),
    }

    # --- PU diagnostics ---------------------------------------------------
    per_class = collections.Counter(d["category_id"] for d in dets)
    scores = np.array([d["score"] for d in dets])
    stats["dets_per_image"] = len(dets) / max(len(predictions), 1)
    stats["n_high_conf"] = int((scores >= 0.5).sum())
    stats["mean_score"] = float(scores.mean())

    if verbose:
        print("\n--- PU diagnostics ---")
        print(f"detections/image      : {stats['dets_per_image']:.1f}")
        print(f"high-confidence (>=.5): {stats['n_high_conf']}")
        print(f"mean score            : {stats['mean_score']:.3f}")
        print(f"distinct classes used : {len(per_class)}")
        if index_to_name:
            name_of = {}
            for c in coco_gt_dict["categories"]:
                name_of[c["id"]] = c.get("name", str(c["id"]))
            print("top predicted classes :")
            for cid, n in per_class.most_common(8):
                print(f"   {name_of.get(cid, cid):26s} {n}")
        print(
            "\nReminder: on an incompletely annotated split a rise in "
            "'false positives'\nmay be recovered unlabeled organisms. Compare "
            "against the fully\nannotated evaluation set before concluding "
            "precision got worse."
        )
    return stats

## 12. Verification

Everything in §3–§4 is checked here against hand-computed expected values, and the
PU criterion against its own properties. This is the `"verify"` stage and it runs
on CPU in seconds. Run it before any GPU stage.

In [ ]:
def run_numpy_tests(verbose=True):
    """Self-contained checks on the NumPy ops. Returns (n_pass, failures)."""
    results, failures = [], []

    def ck(name, cond, detail=""):
        results.append(bool(cond))
        if not cond:
            failures.append(f"{name} {detail}")
        if verbose and not cond:
            print(f"  FAIL  {name}  {detail}")

    def close(a, b, tol=1e-9):
        return abs(float(a) - float(b)) <= tol

    # --- boxes and IoU ---------------------------------------------------
    ck("xywh->xyxy", np.allclose(xywh_to_xyxy([[10, 20, 30, 40]]), [[10, 20, 40, 60]]))
    ck("xyxy->xywh", np.allclose(xyxy_to_xywh([[10, 20, 40, 60]]), [[10, 20, 30, 40]]))
    ck("IoU quarter overlap = 25/175",
       close(iou_matrix([[0, 0, 10, 10]], [[5, 5, 15, 15]])[0, 0], 25 / 175))
    ck("IoU half overlap = 1/3",
       close(iou_matrix([[0, 0, 10, 10]], [[5, 0, 15, 10]])[0, 0], 1 / 3))
    ck("IoU identical = 1", close(iou_matrix([[0, 0, 5, 5]], [[0, 0, 5, 5]])[0, 0], 1.0))
    ck("IoU disjoint = 0", close(iou_matrix([[0, 0, 5, 5]], [[9, 9, 14, 14]])[0, 0], 0.0))
    ck("IoU zero-area is 0 not NaN",
       close(iou_matrix([[5, 5, 5, 5]], [[0, 0, 10, 10]])[0, 0], 0.0))

    # --- Soft-NMS: exact expected decays --------------------------------
    _, s, _ = soft_nms([[0, 0, 10, 10], [0, 0, 10, 10]], [1.0, 1.0],
                       method="gaussian", sigma=0.5)
    ck("gaussian keeps both identical boxes", len(s) == 2, f"{len(s)}")
    ck("gaussian identical decay == exp(-2)", close(s[1], math.exp(-2.0)), f"{s[1]}")

    _, s, _ = soft_nms([[0, 0, 10, 10], [5, 0, 15, 10]], [0.9, 0.8],
                       method="gaussian", sigma=0.5)
    ck("gaussian half-overlap decay",
       close(s[1], 0.8 * math.exp(-((1 / 3) ** 2) / 0.5)), f"{s[1]}")

    _, s, _ = soft_nms([[0, 0, 10, 10], [0, 0, 10, 10]], [1.0, 1.0],
                       method="linear", iou_threshold=0.3)
    ck("linear drops exact duplicate", len(s) == 1, f"{len(s)}")

    _, ss, _ = soft_nms([[0, 0, 10, 10], [2, 2, 12, 12]], [0.9, 0.85], method="gaussian")
    _, sh, _ = soft_nms([[0, 0, 10, 10], [2, 2, 12, 12]], [0.9, 0.85],
                        method="hard", iou_threshold=0.3)
    ck("soft keeps dense neighbour that hard deletes",
       len(ss) == 2 and len(sh) == 1, f"soft={len(ss)} hard={len(sh)}")

    _, s, _ = soft_nms([[0, 0, 5, 5], [50, 50, 55, 55]], [0.7, 0.6])
    ck("disjoint boxes untouched", len(s) == 2 and close(s[0], 0.7) and close(s[1], 0.6))

    _, s, l = soft_nms_per_class([[0, 0, 10, 10], [0, 0, 10, 10]], [0.9, 0.85], [1, 2])
    ck("per-class does not suppress across classes",
       len(s) == 2 and close(s[0], 0.9) and close(s[1], 0.85), f"{s}")

    # --- coordinate mapping ---------------------------------------------
    ck("scale_boxes 3x",
       np.allclose(scale_boxes([[10, 20, 30, 40]], (640, 360), (1920, 1080)),
                   [[30, 60, 90, 120]]))
    ck("scale_boxes anisotropic",
       np.allclose(scale_boxes([[10, 10, 20, 20]], (100, 200), (300, 400)),
                   [[30, 20, 60, 40]]))
    ck("clip to frame", np.allclose(clip_boxes([[-5, -5, 50, 50]], (40, 30)),
                                    [[0, 0, 40, 30]]))

    m = merge_multiscale(
        [{"boxes": [[10, 10, 20, 20]], "scores": [0.9], "labels": [1], "size": (100, 100)},
         {"boxes": [[20, 20, 40, 40]], "scores": [0.8], "labels": [1], "size": (200, 200)}],
        original_size=(100, 100), method="gaussian", sigma=0.5)
    ck("two passes agreeing map onto the same box",
       np.allclose(m["boxes"][0], [10, 10, 20, 20]), f"{m['boxes']}")
    ck("agreeing duplicate decayed not deleted",
       len(m["scores"]) == 2 and close(m["scores"][1], 0.8 * math.exp(-2.0)), f"{m['scores']}")
    ck("out-of-frame box dropped",
       len(merge_multiscale(
           [{"boxes": [[200, 200, 300, 300]], "scores": [0.9], "labels": [1],
             "size": (100, 100)}], original_size=(100, 100))["scores"]) == 0)

    # --- PU weight -------------------------------------------------------
    w = pu_background_weight(np.array([0.5]), w_min=0.25, w_max=1.0,
                             tau=0.5, temperature=0.1)
    ck("w at tau is midpoint 0.625", close(w[0], 0.625), f"{w[0]}")
    w = pu_background_weight(np.linspace(0, 1, 11), w_min=0.25, w_max=1.0,
                             tau=0.5, temperature=0.1)
    ck("w strictly decreasing", bool(np.all(np.diff(w) < 0)))
    ck("w bounded", bool(np.all(w >= 0.25) and np.all(w <= 1.0)))
    ck("w never zero", bool(np.all(w > 0)))
    ck("w_min==w_max is uniform",
       bool(np.allclose(pu_background_weight(np.array([0.0, 0.5, 1.0]),
                                             w_min=1.0, w_max=1.0), 1.0)))

    # --- pseudo-label funnel --------------------------------------------
    gt_b, gt_l = np.array([[0, 0, 10, 10]], float), np.array([1])
    ck("dedup rejects same-class duplicate",
       not bool(dedup_against_gt([[0, 0, 10, 10]], [1], gt_b, gt_l)[0]))
    ck("dedup keeps other class at same place",
       bool(dedup_against_gt([[0, 0, 10, 10]], [2], gt_b, gt_l)[0]))
    ck("dedup keeps distant same-class box",
       bool(dedup_against_gt([[90, 90, 100, 100]], [1], gt_b, gt_l)[0]))
    ck("consistency needs class agreement",
       not bool(consistency_filter(
           [{"boxes": [[0, 0, 10, 10]], "labels": [1]},
            {"boxes": [[0, 0, 10, 10]], "labels": [2]}], min_views=2)[0]))

    res = harvest_pseudo_labels(
        [{"boxes": [[0, 0, 10, 10], [50, 50, 60, 60], [80, 80, 90, 90], [20, 20, 30, 30]],
          "scores": [0.9, 0.8, 0.3, 0.7], "labels": [1, 1, 1, 1]},
         {"boxes": [[50, 50, 60, 60]], "scores": [0.75], "labels": [1]}],
        gt_boxes=gt_b, gt_labels=gt_l, score_threshold=0.6,
        gt_iou_threshold=0.5, consistency_iou=0.6, min_views=2)
    c = res["counts"]
    ck("funnel counts 4/3/2/1",
       (c["candidates"], c["after_confidence"], c["after_gt_dedup"],
        c["after_consistency"]) == (4, 3, 2, 1), f"{c}")
    ck("funnel kept the right box",
       len(res["boxes"]) == 1 and np.allclose(res["boxes"][0], [50, 50, 60, 60]),
       f"{res['boxes']}")
    ck("funnel monotonically non-increasing",
       c["candidates"] >= c["after_confidence"] >= c["after_gt_dedup"]
       >= c["after_consistency"] >= c["kept"], f"{c}")

    # --- audit / clean / split ------------------------------------------
    toy = {
        "images": [{"id": 1, "width": 100, "height": 100},
                   {"id": 2, "width": 100, "height": 100}],
        "categories": [{"id": 1, "name": "crab"}, {"id": 2, "name": "urchin"}],
        "annotations": [
            {"id": 1, "image_id": 1, "category_id": 1, "bbox": [10, 10, 20, 20]},
            {"id": 2, "image_id": 1, "category_id": 1, "bbox": [10, 10, 20, 20]},
            {"id": 3, "image_id": 2, "category_id": 2, "bbox": [0, 0, 0, 10]},
            {"id": 4, "image_id": 2, "category_id": 2, "bbox": [90, 90, 50, 50]},
            {"id": 5, "image_id": 999, "category_id": 1, "bbox": [0, 0, 5, 5]},
            {"id": 6, "image_id": 1, "category_id": 77, "bbox": [0, 0, 5, 5]},
        ],
    }
    r = audit_coco(toy)
    ck("audit finds orphan", 5 in r["orphan_image_id"], f"{r['orphan_image_id']}")
    ck("audit finds unknown class", 6 in r["unknown_category_id"])
    ck("audit finds zero-extent", 3 in r["non_positive_bbox"])
    ck("audit finds out-of-bounds", 4 in r["out_of_bounds_bbox"])
    ck("audit finds exact duplicate", 2 in r["exact_duplicates"])

    n_before = len(toy["annotations"])
    cleaned, summ = clean_coco(toy)
    ck("clean does not mutate input", len(toy["annotations"]) == n_before)
    ck("clean drops orphan", summ.get("dropped_orphan") == 1, f"{summ}")
    ck("clean drops duplicate", summ.get("dropped_duplicate") == 1, f"{summ}")
    ck("clean clips out-of-bounds", summ.get("clipped") == 1, f"{summ}")
    ck("clean is idempotent", clean_coco(cleaned)[1]["kept"] == summ["kept"])

    big = {"images": [{"id": i, "width": 10, "height": 10} for i in range(500)],
           "categories": [{"id": 1, "name": "c"}], "annotations": []}
    tr, va = image_level_split(big, val_fraction=0.2)
    ck("split complete", len(tr) + len(va) == 500)
    ck("split disjoint", not (set(tr) & set(va)))
    ck("split fraction sane", 0.15 < len(va) / 500 < 0.26, f"{len(va)/500}")
    ck("split deterministic", image_level_split(big, val_fraction=0.2) == (tr, va))

    # --- submission ------------------------------------------------------
    rows = build_submission({7: {"boxes": [[10, 20, 40, 60]], "scores": [0.87], "labels": [3]}})
    ck("submission column order exact",
       list(rows[0].keys()) == SUBMISSION_COLUMNS, f"{list(rows[0].keys())}")
    ck("submission emits xywh",
       (rows[0]["bbox_width"], rows[0]["bbox_height"]) == (30.0, 40.0), f"{rows[0]}")
    ck("submission x/y top-left", (rows[0]["bbox_x"], rows[0]["bbox_y"]) == (10.0, 20.0))
    rows = build_submission({1: {"boxes": [[0, 0, 5, 5], [1, 1, 6, 6]],
                                 "scores": [1.5, -0.2], "labels": [1, 2]}})
    ck("submission clamps scores", rows[0]["score"] == 1.0 and rows[1]["score"] == 0.0)
    ck("submission ids unique", len({r["annotation_id"] for r in rows}) == len(rows))

    n_pass = sum(results)
    print(f"  numpy ops: {n_pass}/{len(results)} passed")
    return n_pass, failures


print("=" * 62)
print("VERIFICATION")
print("=" * 62)
n_pass, fails = run_numpy_tests()
if fails:
    print("\nFAILURES:")
    for f in fails:
        print("  -", f)
    raise AssertionError(f"{len(fails)} numpy op checks failed")

if ENV.get("torch", "MISSING").startswith("MISSING"):
    print("\n  (torch missing - skipping PU criterion self-test)")
else:
    print()
    selftest_pu_criterion(verbose=False)

print("\nAll verification passed.")

### 12b. End-to-end rehearsal on synthetic data

A synthetic dataset where we **know** which positives were withheld, so the claim
"Stage 3 recovers missing positives without inventing new ones" becomes a
measurement rather than an assertion. This is the only place in the notebook where
pseudo-label precision can actually be computed, because it is the only place the
withheld labels are known.

In [ ]:
def synthetic_rehearsal(n_images=60, seed=0, verbose=True):
    """
    Build a PU dataset by construction: generate k true objects per frame, then
    annotate only the first. Everything else is a deliberately withheld positive.
    """
    rng = np.random.default_rng(seed)
    images, annotations = [], []
    true_objects, withheld = {}, {}
    ann_id = 1

    for im_id in range(1, n_images + 1):
        images.append({"id": im_id, "width": 640, "height": 480,
                       "file_name": f"{im_id}.png"})
        boxes = []
        for _ in range(int(rng.integers(2, 5))):
            x, y = float(rng.integers(0, 540)), float(rng.integers(0, 380))
            boxes.append([x, y, x + 80, y + 80])
        true_objects[im_id] = boxes
        withheld[im_id] = boxes[1:]
        bb = boxes[0]
        annotations.append({"id": ann_id, "image_id": im_id, "category_id": 1,
                            "bbox": [bb[0], bb[1], bb[2] - bb[0], bb[3] - bb[1]],
                            "area": 6400.0, "iscrowd": 0})
        ann_id += 1

    synth = {"images": images, "categories": [{"id": 1, "name": "organism"}],
             "annotations": annotations}

    rep = audit_coco(synth)
    assert abs(rep["instances_per_image"]["mean"] - 1.0) < 1e-9
    if verbose:
        print(f"synthetic: {n_images} images, "
              f"{rep['instances_per_image']['mean']:.2f} labelled instances/image")
        print(f"withheld positives: {sum(len(v) for v in withheld.values())}")

    tr_ids, va_ids = image_level_split(synth, val_fraction=0.2)

    def detector(im_id, jitter):
        """Finds every true object with jitter, plus one wandering texture artifact."""
        boxes, scores = [], []
        for bb in true_objects[im_id]:
            j = rng.normal(0, jitter, 4)
            boxes.append([bb[0] + j[0], bb[1] + j[1], bb[2] + j[2], bb[3] + j[3]])
            scores.append(float(np.clip(rng.normal(0.85, 0.05), 0, 1)))
        fx = float(rng.integers(0, 500))
        boxes.append([fx, 400.0, fx + 60.0, 460.0])
        scores.append(float(np.clip(rng.normal(0.70, 0.05), 0, 1)))
        return {"boxes": np.array(boxes, float), "scores": np.array(scores, float),
                "labels": np.ones(len(boxes), int)}

    gt_by_image = collections.defaultdict(list)
    for a in synth["annotations"]:
        x, y, w, h = a["bbox"]
        gt_by_image[a["image_id"]].append([x, y, x + w, y + h])

    harvest, kept, recovered, spurious = {}, 0, 0, 0
    for im_id in tr_ids:
        res = harvest_pseudo_labels(
            [detector(im_id, 2.0), detector(im_id, 3.0)],
            gt_boxes=np.array(gt_by_image[im_id], float),
            gt_labels=np.ones(len(gt_by_image[im_id]), int),
            score_threshold=0.6, gt_iou_threshold=0.5,
            consistency_iou=0.5, min_views=2, max_per_image=5)
        harvest[im_id] = res
        kept += res["counts"]["kept"]
        if len(res["boxes"]) and withheld[im_id]:
            hit = iou_matrix(res["boxes"], np.array(withheld[im_id], float)).max(axis=1) >= 0.5
            recovered += int(hit.sum()); spurious += int((~hit).sum())
        else:
            spurious += len(res["boxes"])

    precision = recovered / max(kept, 1)
    total_withheld = sum(len(withheld[i]) for i in tr_ids)
    recall = recovered / max(total_withheld, 1)

    merged, added = merge_pseudo_labels(synth, harvest)
    rep2 = audit_coco(merged)

    if verbose:
        print(f"\npseudo-labels kept    : {kept}")
        print(f"  recovered withheld  : {recovered}")
        print(f"  spurious            : {spurious}")
        print(f"  precision           : {precision:.1%}")
        print(f"  recall of withheld  : {recall:.1%}")
        print(f"\ninstances/image {rep['instances_per_image']['mean']:.2f}"
              f" -> {rep2['instances_per_image']['mean']:.2f} after recovery")

    assert precision > 0.8, f"pseudo-label precision too low: {precision:.3f}"
    assert recovered > 0, "recovered nothing"
    assert rep2["instances_per_image"]["mean"] > rep["instances_per_image"]["mean"]
    assert rep2["non_positive_bbox"] == [] and rep2["orphan_image_id"] == []

    # Two-scale inference + submission on the held-out split.
    preds = {}
    for im_id in va_ids:
        med, hi = detector(im_id, 3.0), detector(im_id, 1.0)
        preds[im_id] = merge_multiscale(
            [{"boxes": med["boxes"], "scores": med["scores"], "labels": med["labels"],
              "size": (640, 480)},
             {"boxes": hi["boxes"], "scores": hi["scores"], "labels": hi["labels"],
              "size": (640, 480)}],
            original_size=(640, 480), method="gaussian", sigma=0.5)

    for p in preds.values():
        b = p["boxes"]
        assert np.all(b[:, 0] >= 0) and np.all(b[:, 1] >= 0)
        assert np.all(b[:, 2] <= 640 + 1e-6) and np.all(b[:, 3] <= 480 + 1e-6)
        assert np.all(b[:, 2] > b[:, 0]) and np.all(b[:, 3] > b[:, 1])

    rows = build_submission(preds)
    assert rows and all(list(r.keys()) == SUBMISSION_COLUMNS for r in rows)
    assert all(0.0 <= r["score"] <= 1.0 for r in rows)
    assert len({r["annotation_id"] for r in rows}) == len(rows)
    if verbose:
        print(f"\nsubmission rows from held-out split: {len(rows)}  (schema OK)")
        print("\nrehearsal passed.")
    return {"precision": precision, "recall": recall, "kept": kept, "rows": len(rows)}


REHEARSAL = synthetic_rehearsal()

## 13. Stage runner

Dispatches on `CFG.stage`. Each GPU stage writes a checkpoint into `work_dir`, so
you can stop after any one of them and resume in a fresh session by changing
`CFG.stage` and re-running the notebook.

In [ ]:
BASE_CKPT = "rtdetr_base.pt"
PU_CKPT = "rtdetr_pu.pt"
PSEUDO_JSON = "pseudo_labels.json"
COCO_PU_JSON = "coco_with_pseudo.json"

def criterion_config_from(cfg, pu_on):
    return PULossConfig(
        num_classes=cfg.num_classes,
        pu_w_min=cfg.pu_w_min, pu_w_max=cfg.pu_w_max,
        pu_tau=cfg.pu_tau, pu_temperature=cfg.pu_temperature,
        pu_disabled=not pu_on,
        pseudo_label_weight=cfg.pseudo_label_weight,
    )


def prepare_images_for(ids, coco, cfg):
    """Resolve imagery for `ids`, then return only the ids actually on disk."""
    subset = subset_coco(coco, ids)
    attached = find_attached_image_dir(
        cfg.input_roots, [im.get("file_name", "") for im in subset["images"][:20]]
    )
    cache = Path(attached) if attached else Path(cfg.work_dir) / "images"
    if attached:
        print(f"using attached image directory: {cache}")
    ok, failed = ensure_images(subset, cache, workers=cfg.download_workers)
    # Propagate the resolved paths back onto the parent COCO dict.
    resolved = {im["id"]: im.get("_local_path") for im in subset["images"]}
    for im in coco["images"]:
        if im["id"] in resolved:
            im["_local_path"] = resolved[im["id"]]
    if failed:
        print(f">>> {len(failed)} frames unavailable and will be skipped")
    return [i for i in ids if i in ok]


if CFG.stage == "verify":
    print("\nVerification stage complete. "
          "Set CFG.stage='audit' next, then the GPU stages.")

elif CFG.stage == "audit":
    if raw_train is None:
        print("no training annotations found - nothing to audit")
    else:
        out = Path(CFG.work_dir) / "stage1_audit.json"
        out.write_text(json.dumps(
            {"audit": {k: (v if isinstance(v, (int, float, dict, str)) else len(v))
                       for k, v in AUDIT.items()},
             "clean_summary": clean_summary,
             "n_train": len(train_ids), "n_val": len(val_ids)}, indent=2, default=str))
        print(f"\nStage 1 report -> {out}")
        print("Set CFG.stage='train_base' and enable GPU + Internet next.")

elif CFG.stage in {"train_base", "harvest", "train_pu", "infer"}:
    import torch
    assert torch.cuda.is_available(), "this stage needs a GPU"
    assert raw_train is not None, "training annotations not found"

    cat_id_to_index, index_to_cat_id, index_to_name = build_category_maps(coco_clean)
    assert len(cat_id_to_index) == CFG.num_classes, (
        f"config says {CFG.num_classes} classes but the data has "
        f"{len(cat_id_to_index)} - fix CFG.num_classes"
    )

    use_train = train_ids if not CFG.max_images else train_ids[: CFG.max_images]
    use_val = val_ids if not CFG.max_images else val_ids[: max(CFG.max_images // 4, 1)]
    print(f"using {len(use_train)} train / {len(use_val)} val images"
          + ("" if not CFG.max_images else f"  (capped by max_images={CFG.max_images})"))

    if CFG.stage == "train_base":
        use_train = prepare_images_for(use_train, coco_clean, CFG)
        use_val = prepare_images_for(use_val, coco_clean, CFG)
        print("\n--- Stage 2: baseline detector (PU loss OFF) ---")
        model, hist, ckpt = train_one_run(
            coco_clean, use_train, use_val, CFG,
            criterion_config_from(CFG, pu_on=False), BASE_CKPT)
        print(f"\nbaseline checkpoint: {ckpt}")
        print("Next: CFG.stage='harvest'")

    elif CFG.stage == "harvest":
        use_train = prepare_images_for(use_train, coco_clean, CFG)
        ckpt = Path(CFG.work_dir) / BASE_CKPT
        assert ckpt.is_file(), f"run stage 'train_base' first ({ckpt} missing)"
        device = torch.device("cuda")
        model = build_model(CFG.num_classes, CFG.model_name, device)
        model.load_state_dict(load_checkpoint(ckpt, device)["model"])
        print("\n--- Stage 3: conservative pseudo-label recovery ---")
        harvest, totals = harvest_over_dataset(
            model, coco_clean, use_train, CFG, index_to_cat_id)
        coco_pu, added = merge_pseudo_labels(coco_clean, harvest)
        print(f"\nadded {added} pseudo-labels")
        rep_before = audit_coco(coco_clean)["instances_per_image"]["mean"]
        rep_after = audit_coco(coco_pu)["instances_per_image"]["mean"]
        print(f"instances/image {rep_before:.2f} -> {rep_after:.2f}")
        (Path(CFG.work_dir) / COCO_PU_JSON).write_text(json.dumps(coco_pu))
        (Path(CFG.work_dir) / PSEUDO_JSON).write_text(json.dumps(
            {str(k): {"boxes": v["boxes"].tolist(), "scores": v["scores"].tolist(),
                      "labels": v["labels"].tolist()} for k, v in harvest.items()}))
        print(f"-> {Path(CFG.work_dir) / COCO_PU_JSON}")
        print("Next: CFG.stage='train_pu'")

    elif CFG.stage == "train_pu":
        pu_json = Path(CFG.work_dir) / COCO_PU_JSON
        if pu_json.is_file():
            coco_for_training = json.loads(pu_json.read_text())
            n_pseudo = sum(1 for a in coco_for_training["annotations"] if a.get("is_pseudo"))
            print(f"training on pseudo-augmented data ({n_pseudo} pseudo-labels)")
        else:
            coco_for_training = coco_clean
            print(">>> no pseudo-labels found; training on cleaned data only")
        use_train = prepare_images_for(use_train, coco_for_training, CFG)
        use_val = prepare_images_for(use_val, coco_for_training, CFG)
        print("\n--- Stages 3+4: retrain with PU-aware background loss ---")
        cc = criterion_config_from(CFG, pu_on=not CFG.pu_disabled)
        print(f"PU: w_min={cc.pu_w_min} w_max={cc.pu_w_max} "
              f"tau={cc.pu_tau} T={cc.pu_temperature} disabled={cc.pu_disabled}")
        model, hist, ckpt = train_one_run(
            coco_for_training, use_train, use_val, CFG, cc, PU_CKPT,
            init_from=Path(CFG.work_dir) / BASE_CKPT)
        print(f"\nPU checkpoint: {ckpt}")
        print("Next: CFG.stage='infer'")

    elif CFG.stage == "infer":
        ckpt = Path(CFG.work_dir) / PU_CKPT
        if not ckpt.is_file():
            ckpt = Path(CFG.work_dir) / BASE_CKPT
            print(f">>> PU checkpoint missing, falling back to {ckpt}")
        assert ckpt.is_file(), "no checkpoint - run a training stage first"
        device = torch.device("cuda")
        model = build_model(CFG.num_classes, CFG.model_name, device)
        model.load_state_dict(load_checkpoint(ckpt, device)["model"])

        print("\n--- Stage 5: two-scale inference + Soft-NMS ---")
        print(f"scales {CFG.infer_scales}  soft-nms {CFG.softnms_method}"
              f" sigma={CFG.softnms_sigma}")

        # Validation pass, for the diagnostics (mAP here is a relative signal only).
        use_val = prepare_images_for(use_val, coco_clean, CFG)
        val_preds = predict_two_scale(model, coco_clean, use_val, CFG, index_to_cat_id)
        print(f"\nvalidation predictions on {len(val_preds)} images")
        val_stats = evaluate_map(subset_coco(coco_clean, use_val), val_preds,
                                 index_to_name=index_to_name)

        # Test pass -> submission.
        if test_json_path is not None:
            raw_test = load_coco(test_json_path)
            test_ids = [im["id"] for im in raw_test["images"]]
            if CFG.max_images:
                test_ids = test_ids[: CFG.max_images]
            test_ids = prepare_images_for(test_ids, raw_test, CFG)
            test_preds = predict_two_scale(model, raw_test, test_ids, CFG, index_to_cat_id)
            rows = build_submission(test_preds)
            import csv
            sub = Path(CFG.work_dir) / "submission.csv"
            with open(sub, "w", newline="") as fh:
                wr = csv.DictWriter(fh, fieldnames=SUBMISSION_COLUMNS)
                wr.writeheader()
                wr.writerows(rows)
            print(f"\nsubmission: {len(rows)} rows -> {sub}")
            print(f"  images with detections: {len(test_preds)}/{len(test_ids)}")
        else:
            print("\nno test annotations found - skipping submission")

else:
    raise ValueError(f"unknown stage {CFG.stage!r}")

## 14. Ablations worth running

The pipeline was kept small on purpose: if a number moves, you want to know
*which idea* moved it. Four runs answer that, and they are all the same code with
a different config:

| Run | Change | Question it answers |
|---|---|---|
| A | `pu_disabled=True`, no pseudo-labels | Baseline |
| B | `pu_disabled=True` + pseudo-labels | Did Stage 3 alone help? |
| C | `pu_disabled=False`, no pseudo-labels | Did the PU loss alone help? |
| D | Both | Do they compose, or overlap? |

Then sweep `pu_w_min` over `{0.1, 0.25, 0.5, 0.75}` at fixed everything else.
Expect a precision–recall trade-off rather than a free win — and expect
`pu_w_min=0` to fail loudly, which is the point.

Two cautions when reading the results:

- **Validation mAP on the training split is partially wrong**, because that split
  is also incompletely annotated. Treat it as a relative signal between runs.
- **A rise in high-confidence "false positives" is ambiguous.** It is the
  expected signature of recovering unlabeled organisms *and* of the background
  weight being too low. Only the fully annotated evaluation set separates the two.